# Midline verification: `fish_midline_from_mask`

This notebook checks the midline function used by `eod_two_layer_pipeline_clean.ipynb`
(cell 5) on synthetic fish with a **known** midline and on the **real** tracking masks, compared
against the tracked keypoints in the CSV.

| § | Question |
|---|---|
| 1 | Are the coordinate conventions (rows/cols, `/(P-1)` scaling) consistent with the solver grid and the electrodes? |
| 2 | Synthetic fish: what the fit looks like for straight, C-, S- and strongly bent bodies |
| 3 | Synthetic accuracy table (error in px vs the true midline) |
| 4 | Step 4 (averaging pixels with equal `t`): does it ever actually merge anything? |
| 5 | Smoothing: the default is effectively a cubic polynomial; smaller `smooth` overfits |
| 6 | Point spacing along the curve, and the arc-length resampling in `multipole_sources` |
| 7 | Real data: head/tail orientation, end-point distances, mask contamination, per-frame inspector |
| 8 | A candidate fixed version (`fish_midline_v2`) evaluated on the same tests, for comparison only |
| 9 | Alternative: tracing midline (`fish_midline_trace`) that follows any bend |
| 10 | PCA vs tracing on synthetic fish (incl. 200–320° curls), start-point and parameter checks |
| 11 | PCA vs tracing on real masks, side-by-side galleries, run time |
| 12 | Principal curve, Isomap, and Denis's keypoint method (EODAssignment) |
| 13 | All five methods on synthetic fish |
| 14 | All five methods on real masks: checks, pairwise agreement, galleries |
| 15 | Fin removal (width envelope) and the measured geometry of real fins |
| 16 | Synthetic fish with realistic fins; every method with vs without fin removal |
| 17 | Limiting zig-zags: penalty on changes in bending |
| 18 | Mask-based Denis: keypoints from the mask, 4 vs 8 (vs 12) points |
| 19 | Final comparison of all methods in their best form |
| 20 | Five approaches head to head (best two with / without fin removal, Denis with SLEAP keypoints): tables + 20 real fish |

All distances are in **video pixels** unless stated. The fish are ~150–200 px long and ~15–25 px wide.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import zoom, label
from scipy.interpolate import UnivariateSpline

MASK_PATH = 'data/video2024-02-20T21_16_26__223e__59997__masks_packed.mgz_masks_first_1000_frames.npy'
CSV_PATH  = 'data/Data_4fish_2024-02-20T21_16_26.csv'
STRIDE    = 5          # real-data checks use every STRIDE-th mask frame (1 = all 1001 frames, slower)
plt.rcParams['figure.dpi'] = 100

## 0. Code under test

`fish_midline_from_mask` and `multipole_sources` are copied **verbatim** from
`eod_two_layer_pipeline_clean.ipynb` (cell 5). If you change them there, paste the new version here.

`midline_internals` repeats the same steps but also returns the intermediate quantities
(axis `v`, coordinates `t, w`, spline knots …) so we can look inside. The assert checks that it
reproduces the function exactly.

In [ ]:
def fish_midline_from_mask(mask, n_ctrl=200, smooth=None):
    '''(P,P) bool -> smooth curved midline (n_ctrl,2) in [0,1], oriented tail->head.'''
    xs, ys = np.nonzero(mask)
    pts = np.column_stack([xs, ys]).astype(float); c = pts.mean(0)
    v = np.linalg.svd(pts - c, full_matrices=False)[2][0]
    perp = np.array([-v[1], v[0]])
    t = (pts - c) @ v
    w = (pts - c) @ perp

    # average perpendicular offset per unique t (spline needs strictly increasing x)
    order = np.argsort(t); t_s, w_s = t[order], w[order]
    tu, inv, cnt = np.unique(np.round(t_s, 6), return_inverse=True, return_counts=True)
    wu = np.bincount(inv, weights=w_s) / cnt
    L = t_s.max() - t_s.min()
    if smooth is None:
        smooth = len(tu) * (0.05 * L) ** 2             # gentle default
    spl = UnivariateSpline(tu, wu, k=3, s=smooth, w=np.sqrt(cnt))
    ts = np.linspace(t_s.min(), t_s.max(), n_ctrl)
    curve = c + np.outer(ts, v) + np.outer(spl(ts), perp)

    # orient tail->head: the head end is bulkier (more pixels) -> that half goes last
    t_mid = 0.5 * (t.min() + t.max())
    if np.count_nonzero(t < t_mid) > np.count_nonzero(t >= t_mid):
        curve = curve[::-1]

    nx, ny = mask.shape
    return curve / np.array([nx - 1, ny - 1])

def multipole_sources(midline, n, I0=1.0, exp_length_scale=0):
    midline = np.asarray(midline, float)
    seg = np.linalg.norm(np.diff(midline, axis=0), axis=1)
    s = np.concatenate([[0.0], np.cumsum(seg)]); s /= s[-1]
    targets = np.arange(n + 1) / n
    pos = np.column_stack([np.interp(targets, s, midline[:, 0]),
                            np.interp(targets, s, midline[:, 1])])
    m = np.full(n + 1, 1.0 / n); m[0] = -1.0
    return pos, m * I0

In [ ]:
def midline_internals(mask, n_ctrl=200, smooth=None):
    '''Same steps as fish_midline_from_mask, returning the intermediates (pixel units).'''
    xs, ys = np.nonzero(mask)
    pts = np.column_stack([xs, ys]).astype(float); c = pts.mean(0)
    v = np.linalg.svd(pts - c, full_matrices=False)[2][0]
    perp = np.array([-v[1], v[0]])
    t = (pts - c) @ v; w = (pts - c) @ perp
    order = np.argsort(t); t_s, w_s = t[order], w[order]
    tu, inv, cnt = np.unique(np.round(t_s, 6), return_inverse=True, return_counts=True)
    wu = np.bincount(inv, weights=w_s) / cnt
    L = t_s.max() - t_s.min()
    s = len(tu) * (0.05 * L) ** 2 if smooth is None else smooth
    spl = UnivariateSpline(tu, wu, k=3, s=s, w=np.sqrt(cnt))
    ts = np.linspace(t_s.min(), t_s.max(), n_ctrl)
    curve = c + np.outer(ts, v) + np.outer(spl(ts), perp)
    t_mid = 0.5 * (t.min() + t.max())
    n_low, n_high = np.count_nonzero(t < t_mid), np.count_nonzero(t >= t_mid)
    flipped = n_low > n_high
    if flipped: curve = curve[::-1]
    return dict(c=c, v=v, perp=perp, t=t, w=w, tu=tu, wu=wu, cnt=cnt, L=L, s=s, spl=spl,
                n_knots=len(spl.get_knots()), n_unique=len(tu), n_pix=len(t),
                n_low=n_low, n_high=n_high, flipped=flipped, curve_px=curve)

def to_px(curve01, P):
    return curve01 * (P - 1)

def dist_to_poly(p, poly):
    '''distance from each point in p to the nearest point of the densely sampled polyline poly'''
    return np.min(np.linalg.norm(poly[None] - p[:, None], axis=2), axis=1)

_m = np.zeros((60, 60), bool); _m[20:28, 5:55] = True; _m[18:30, 35:50] = True
assert np.allclose(to_px(fish_midline_from_mask(_m), 60), midline_internals(_m)['curve_px'])
print('midline_internals reproduces fish_midline_from_mask exactly')

## 1. Coordinate conventions

**1a.** `np.nonzero(mask)` returns `(row, col)`. The function calls them `xs, ys`, so its first
coordinate is the image **row** (the CSV's `y`).

In [ ]:
m = np.zeros((5, 8), bool); m[1, 6] = True        # row 1, column 6
print('np.nonzero ->', [int(a[0]) for a in np.nonzero(m)], ' (row, col)')

**1b.** The midline is scaled by `/(P-1)`, so pixel *index* 0 maps to 0 and *P-1* maps to 1
("corner-aligned"). The occupancy grid is made with `zoom(mask, N/P)` and the solver grid is
`linspace(0, 1, N)`. `scipy.ndimage.zoom` (default `grid_mode=False`) also aligns the corner pixel
centres, so grid node `j` sits at pixel `j·(P-1)/(N-1)`, i.e. normalised position `j/(N-1)`.
Below: a real fish mask's centroid computed in normalised units both ways should agree to a small
fraction of a grid cell. The last line shows the shift a *cell-centred* `(i+0.5)/P` convention
would have introduced instead.

In [ ]:
M = np.load(MASK_PATH, mmap_mode='r')          # (n_fish, n_frames, P, P)
n_fish, n_frames, P, _ = M.shape
print('masks:', M.shape)

fm = np.asarray(M[1, 0])
for N in (200, 400, 912):
    zg = zoom(fm.astype(float), N / P, order=1)[:N, :N]
    gi, gj = np.mgrid[0:N, 0:N]
    c_grid = np.array([(zg * gi).sum(), (zg * gj).sum()]) / zg.sum() / (N - 1)
    c_mask = np.array(np.nonzero(fm)).mean(1) / (P - 1)
    print(f'N={N:4d}: zoom grid shape {zg.shape}, centroid diff = {np.abs(c_grid - c_mask).max()*(N-1):.3f} grid cells')
c_cell = (np.array(np.nonzero(fm)).mean(1) + 0.5) / P
print(f'cell-centred convention would shift it by {np.abs(c_cell - c_mask).max()*(P-1):.3f} px')

**1c.** The electrodes are loaded as `[y/(P-1), x/(P-1)]`, i.e. (row, col), the same as the
midline. To check this ordering against the masks independently, map the CSV keypoints the same
way (`(y, x)` → (row, col)) and check that they land on a fish mask. If the axes were swapped,
almost none would.

In [ ]:
df = pd.read_csv(CSV_PATH).groupby('Frame').first()       # several EOD rows per frame -> one
KP = ['chin', 'head', 'middle', 'tail']

def keypoint(r, part, k):
    '''CSV keypoint of CSV fish k as (row, col) mask coordinates'''
    return np.array([r[f'{part}_y_{k}'], r[f'{part}_x_{k}']], float)

hits = {'(y, x)  [used]': [], '(x, y)  [swapped]': []}
for fr in range(0, n_frames, 50):
    if fr + 1 not in df.index: continue      # not every frame has a CSV row
    r = df.loc[fr + 1]                       # CSV Frame is 1-indexed
    union = np.asarray(M[:, fr]).any(0)
    for k in range(1, 5):
        for part in ('head', 'middle'):
            p = keypoint(r, part, k)
            if np.isnan(p).any(): continue
            for name, q in (('(y, x)  [used]', p), ('(x, y)  [swapped]', p[::-1])):
                i, j = np.round(q).astype(int)
                hits[name].append(0 <= i < P and 0 <= j < P and union[max(i-2,0):i+3, max(j-2,0):j+3].any())
for name, h in hits.items():
    print(f'{name}: {np.mean(h)*100:5.1f}% of head/middle keypoints within 2 px of a mask  (n={len(h)})')

## 2. Synthetic fish with a known midline

`make_fish` builds a midline of length `Lf` = 180 px with a chosen total bend, then paints a body
around it as a union of disks. The half-width grows from 1.5 px at the tail to 10.5 px at 75 % of
the length and narrows to 0 at the snout (head bulkier than tail, like the real fish). This
width profile is an assumption, not measured from the real fish. `kind='C'` bends with constant
curvature; `kind='S'` bends one way and then the other. The true midline is returned
tail → head, in pixel (row, col) coordinates.

In [ ]:
PS, Lf = 912, 180.0

def make_fish(theta_deg, bend, kind='C', center=(456.3, 456.7), P=PS):
    s = np.linspace(0, Lf, 2000)
    if kind == 'C':
        kappa = np.deg2rad(bend) / Lf * np.ones_like(s)
    else:
        kappa = np.deg2rad(bend) / Lf * 2 * np.cos(2 * np.pi * s / Lf)
    ang = np.concatenate([[0], np.cumsum(kappa[:-1] * np.diff(s))]) + np.deg2rad(theta_deg)
    ds = np.diff(s, prepend=0)
    mid = np.column_stack([np.cumsum(np.cos(ang) * ds), np.cumsum(np.sin(ang) * ds)])
    mid += np.asarray(center) - mid.mean(0)
    u = s / Lf
    half = 1.5 + 9.0 * np.clip(u / 0.75, 0, 1) ** 1.2
    half = np.where(u > 0.85, half * np.sqrt(np.clip((1 - u) / 0.15, 0, 1)) + 1e-9, half)
    lo, hi = mid.min(0).astype(int) - 15, mid.max(0).astype(int) + 15
    gx, gy = np.mgrid[lo[0]:hi[0], lo[1]:hi[1]]
    sub = np.zeros(gx.shape, bool)
    for (mx, my), h in zip(mid[::4], half[::4]):
        sub |= (gx - mx) ** 2 + (gy - my) ** 2 <= h ** 2
    mask = np.zeros((P, P), bool); mask[lo[0]:hi[0], lo[1]:hi[1]] = sub
    return mid, mask

def score(cur_px, mid):
    '''errors of a fitted curve vs the true midline, in px'''
    d_fit = dist_to_poly(cur_px, mid)          # fit -> truth
    d_true = dist_to_poly(mid[::5], cur_px)    # truth -> fit (catches a fit that is too short)
    return dict(mean_err=d_fit.mean(), max_err=max(d_fit.max(), d_true.max()),
                tail_end_err=np.linalg.norm(cur_px[0] - mid[0]),
                head_end_err=np.linalg.norm(cur_px[-1] - mid[-1]),
                orient_ok=np.linalg.norm(cur_px[-1] - mid[-1]) < np.linalg.norm(cur_px[0] - mid[-1]))

def show_fit(ax, mask, curves, truth=None, pad=12, title=''):
    xs, ys = np.nonzero(mask)
    r0, r1, c0, c1 = xs.min() - pad, xs.max() + pad, ys.min() - pad, ys.max() + pad
    ax.imshow(mask[r0:r1, c0:c1], cmap='gray', extent=(c0 - .5, c1 - .5, r1 - .5, r0 - .5))
    if truth is not None:
        ax.plot(truth[:, 1], truth[:, 0], 'g--', lw=2, label='true midline')
    for (cur, style, lab) in curves:
        ax.plot(cur[:, 1], cur[:, 0], style, lw=1.5, label=lab)
        ax.plot(cur[-1, 1], cur[-1, 0], style[0] + '^', ms=9)
    ax.set_title(title, fontsize=9); ax.legend(fontsize=7, loc='best')

SHAPES = [('C', 0), ('C', 40), ('C', 90), ('S', 30), ('S', 60), ('C', 150)]
fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for ax, (kind, bend) in zip(axs.flat, SHAPES):
    mid, mask = make_fish(30, bend, kind)
    cur = to_px(fish_midline_from_mask(mask), PS)
    sc = score(cur, mid)
    show_fit(ax, mask, [(cur, 'y-', 'fit (triangle = "head")')], truth=mid,
             title=f"{kind}-bend {bend}°: mean {sc['mean_err']:.2f}px, max {sc['max_err']:.2f}px")
plt.tight_layout(); plt.show()

## 3. Synthetic accuracy table

Every shape at 5 rotations. Columns:
`mean_err`/`max_err`: distance between the fit and the true midline (max is two-sided);
`tail_end_err`/`head_end_err`: distance of the curve's first/last point to the true tail/snout;
`orient_ok`: head end placed at the head; `n_unique/n_pix` (see §4); `n_knots` (see §5;
2 = only the two end knots, i.e. a single cubic).

In [ ]:
rows = []
for kind, bend in SHAPES:
    for theta in (0, 17, 30, 45, 90):
        mid, mask = make_fish(theta, bend, kind)
        I = midline_internals(mask)
        rows.append(dict(shape=f'{kind}{bend}', rot=theta, **score(I['curve_px'], mid),
                         n_unique=I['n_unique'], n_pix=I['n_pix'], n_knots=I['n_knots']))
synth = pd.DataFrame(rows)
display(synth.round(2))
display(synth.groupby('shape', sort=False)[['mean_err', 'max_err', 'tail_end_err', 'head_end_err']].mean().round(2))

## 4. Step 4: averaging pixels that share the same `t`

The code averages `w` over pixels with identical `t` (after rounding to 1e-6) and weights the
spline by `sqrt(count)`. In the table above, `n_unique == n_pix` in every case: the PCA axis is
never exactly parallel to the pixel grid, so no two pixels share a `t`, nothing is averaged and
all weights are 1. The spline is fitted to the raw pixel cloud.

The only way to trigger the merge is a fish whose PCA axis is *exactly* on a grid axis or
diagonal. Below we try to construct one (straight fish, integer centre, rotation 0/90).
When merging does happen, `len(tu)` shrinks, and since the default `smooth` is proportional to
`len(tu)`, the smoothing strength changes too. Compare `n_knots` and the errors.

In [ ]:
rows = []
for theta in (0, 90, 45):
    for center in ((456.3, 456.7), (456.0, 456.0)):
        mid, mask = make_fish(theta, 0, 'C', center=center)
        I = midline_internals(mask)
        rows.append(dict(rot=theta, center=center, v=np.round(I['v'], 6), n_unique=I['n_unique'],
                         n_pix=I['n_pix'], n_knots=I['n_knots'], **score(I['curve_px'], mid)))
display(pd.DataFrame(rows).round(3))

## 5. Smoothing

`UnivariateSpline` adds knots until `sum((w_i·(y_i − spl(x_i)))²) ≤ s`. The default
`s = n·(0.05·L)²` allows an RMS residual of 0.05·L ≈ 9 px. The scatter of the pixel cloud across the
body width is only ~5 px, so a single cubic already satisfies it and the spline stops at
**no interior knots**. The left panel shows the pixel cloud in `(t, w)` with the fitted cubic.

The sweep then varies the smoothing as `s = n·(frac·L)²`:
* **raw**: the current method (every pixel is a data point);
* **binned**: average `t` and `w` in 1-px slices along `t` first (the centroid of each
  cross-section), then fit those averages.

The dashed line marks the default `frac = 0.05`.

In [ ]:
mid, mask = make_fish(30, 60, 'S')
I = midline_internals(mask)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(I['t'], I['w'], ',', color='0.6', label='pixels (t, w)')
tt = np.linspace(I['t'].min(), I['t'].max(), 400)
ax.plot(tt, I['spl'](tt), 'r', label=f"default spline ({I['n_knots']} knots)")
rel = (mid - I['c'])
ax.plot(rel @ I['v'], rel @ I['perp'], 'g--', label='true midline')
ax.set_xlabel('t (px along PCA axis)'); ax.set_ylabel('w (px)'); ax.legend(); ax.set_title('S-bend 60°')
plt.show()
print(f"pixel-cloud RMS around the default fit: {np.sqrt(np.mean((I['w'] - I['spl'](I['t']))**2)):.2f} px;"
      f"  default allows RMS {0.05 * I['L']:.2f} px")

In [ ]:
def fit_tw(t, w, frac, binned):
    '''returns (spline, n_knots); raw = current method, binned = 1-px slice centroids'''
    if binned:
        b = np.floor(t - t.min()).astype(int); cnt = np.bincount(b); k = cnt > 0
        tu = np.bincount(b, t)[k] / cnt[k]; wu = np.bincount(b, w)[k] / cnt[k]; cnt = cnt[k]
    else:
        o = np.argsort(t)
        tu, inv, cnt = np.unique(np.round(t[o], 6), return_inverse=True, return_counts=True)
        wu = np.bincount(inv, weights=w[o]) / cnt
    L = t.max() - t.min()
    spl = UnivariateSpline(tu, wu, k=3, s=len(tu) * (frac * L) ** 2, w=np.sqrt(cnt))
    return spl, len(spl.get_knots())

def curve_from(I, spl, n=200):
    ts = np.linspace(I['t'].min(), I['t'].max(), n)
    return I['c'] + np.outer(ts, I['v']) + np.outer(spl(ts), I['perp'])

fracs = np.logspace(-3, -1, 25)
cases = [('C', 90), ('S', 30), ('S', 60)]
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for kind, bend in cases:
    mid, mask = make_fish(30, bend, kind); I = midline_internals(mask)
    for binned, ls in ((False, '-'), (True, '--')):
        kn, err = [], []
        for f in fracs:
            spl, nk = fit_tw(I['t'], I['w'], f, binned)
            kn.append(nk); err.append(score(curve_from(I, spl), mid)['max_err'])
        lab = f"{kind}{bend} {'binned' if binned else 'raw'}"
        axs[0].loglog(fracs, kn, ls, marker='.', label=lab); axs[1].loglog(fracs, err, ls, marker='.', label=lab)
for ax, yl in zip(axs, ('number of knots', 'max error vs truth (px)')):
    ax.axvline(0.05, color='k', ls=':', lw=1); ax.set_xlabel('frac  (s = n·(frac·L)²)'); ax.set_ylabel(yl); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

Things to look for: with **raw** fitting, the knot count jumps from 2 (a single cubic) to
hundreds just below `frac ≈ 0.03`, and the error then gets *worse* rather than better (~10 px,
or ~40 px for the S30 fish) because the spline starts following individual edge pixels. Raw
fitting has no setting that is both flexible and stable. With **binned** fitting, everything
below `frac ≈ 0.005` gives ~1–2 px max error for all three shapes. At the default `frac = 0.05`
both versions give the same single cubic.

Below: the default fit vs a binned fit with `frac = 0.005` on the S-bends.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, (kind, bend) in zip(axs, (('S', 30), ('S', 60))):
    mid, mask = make_fish(30, bend, kind); I = midline_internals(mask)
    spl, nk = fit_tw(I['t'], I['w'], 0.005, True)
    cb = curve_from(I, spl)
    show_fit(ax, mask, [(I['curve_px'], 'y-', f"default: max {score(I['curve_px'], mid)['max_err']:.1f}px"),
                        (cb, 'c-', f"binned 0.005: max {score(cb, mid)['max_err']:.1f}px")],
             truth=mid, title=f'{kind}-bend {bend}°')
plt.tight_layout(); plt.show()

## 6. Point spacing along the curve

`ts` is uniform along the PCA axis, not along the curve, so neighbouring points are further apart
where the body is steep relative to the axis. `multipole_sources` re-interpolates by arc
length, so its sources should be evenly spaced regardless. The ratio max/min spacing should
be ≈1 after resampling.

In [ ]:
rows = []
for kind, bend in SHAPES:
    mid, mask = make_fish(30, bend, kind)
    cur = fish_midline_from_mask(mask)
    sp_mid = np.linalg.norm(np.diff(cur, axis=0), axis=1)
    pos, cur_I = multipole_sources(cur, n=100)
    sp_src = np.linalg.norm(np.diff(pos, axis=0), axis=1)
    rows.append(dict(shape=f'{kind}{bend}', midline_spacing_max_over_min=sp_mid.max() / sp_mid.min(),
                     source_spacing_max_over_min=sp_src.max() / sp_src.min(),
                     total_current=cur_I.sum(), sink=cur_I[0]))
display(pd.DataFrame(rows).round(4))

## 7. Real data

For every `STRIDE`-th frame and every non-empty mask, the matching CSV fish is the one whose
`head` and `tail` keypoints are closest to that mask's pixels (sum of the two distances; matches
further than 10 px are skipped). For each mask we record:

* `orient_ok`: the fitted head end is nearer the tracked head and the fitted tail end nearer the
  tracked tail (compares both pairings, so it doesn't depend on how far the ends are);
* `tail_end_err`: fitted tail end to tracked `tail` keypoint; `head_end_err`: fitted head end to
  tracked `chin` keypoint (the front-most keypoint);
* `n_components`: connected pieces in the mask (more than 1 = contamination or fragments);
* `n_knots`, `n_pix`, the pixel counts in each half (`n_low`, `n_high`), and the matched CSV fish.

The tracked keypoints are not ground truth either; small end-point distances are expected.

In [ ]:
def match_csv_fish(mask, r):
    xs, ys = np.nonzero(mask); pix = np.column_stack([xs, ys])
    best = None
    for k in range(1, 5):
        h, t = keypoint(r, 'head', k), keypoint(r, 'tail', k)
        if np.isnan(h).any() or np.isnan(t).any(): continue
        d = np.min(np.linalg.norm(pix - h, axis=1)) + np.min(np.linalg.norm(pix - t, axis=1))
        if best is None or d < best[0]: best = (d, k)
    return best

def turning_deg(cur):
    '''total absolute turning of the curve in degrees (a smooth fish: tens of degrees;
       a curve that wiggles through edge pixels: thousands)'''
    d = np.diff(cur, axis=0); a = np.unwrap(np.arctan2(d[:, 1], d[:, 0]))
    return np.degrees(np.abs(np.diff(a)).sum())

def end_metrics(cur, head, tail, chin):
    ok = (np.linalg.norm(cur[-1] - head) + np.linalg.norm(cur[0] - tail)
          < np.linalg.norm(cur[0] - head) + np.linalg.norm(cur[-1] - tail))
    return dict(orient_ok=ok, tail_end_err=np.linalg.norm(cur[0] - tail),
                head_end_err=np.linalg.norm(cur[-1] - chin) if not np.isnan(chin).any() else np.nan)

recs = []
for fr in range(0, n_frames, STRIDE):
    if fr + 1 not in df.index: continue
    r = df.loc[fr + 1]
    masks = np.asarray(M[:, fr])
    for f in range(n_fish):
        m = masks[f]
        if m.sum() < 50: continue
        best = match_csv_fish(m, r)
        if best is None or best[0] > 10:
            recs.append(dict(frame=fr, fish=f, matched=False)); continue
        k = best[1]
        I = midline_internals(m)
        rec = dict(frame=fr, fish=f, matched=True, csv_fish=k, match_dist=best[0],
                   n_pix=I['n_pix'], n_knots=I['n_knots'], n_low=I['n_low'], n_high=I['n_high'],
                   n_components=label(m)[1], length_px=I['L'], turning_deg=turning_deg(I['curve_px']),
                   **end_metrics(I['curve_px'], keypoint(r, 'head', k), keypoint(r, 'tail', k), keypoint(r, 'chin', k)))
        recs.append(rec)
real = pd.DataFrame(recs)
rm = real[real.matched]
print(f"{len(real)} masks checked, {len(rm)} matched to a CSV fish, {(~real.matched).sum()} unmatched")
print(f"orientation correct: {rm.orient_ok.sum()}   flipped: {(~rm.orient_ok).sum()}  ({(~rm.orient_ok).mean()*100:.2f}%)")
print('knot counts:', rm.n_knots.value_counts().to_dict())
print('masks with >1 connected component:', (rm.n_components > 1).sum())

### 7a. Orientation failures

A flipped midline puts the sink (tail) at the head, which reverses that fish's simulated field.
Each panel: yellow = fit, red triangle = the end the function calls the head, cyan/magenta =
tracked head/tail, white = tracked chin.

In [ ]:
def show_real(ax, fr, f, curves=None, pad=12):
    m = np.asarray(M[f, fr]); r = df.loc[fr + 1]
    if curves is None:
        curves = [(to_px(fish_midline_from_mask(m), P), 'y-', 'fit (triangle = "head")')]
    show_fit(ax, m, curves, pad=pad)
    best = match_csv_fish(m, r)
    if best is not None and best[0] <= 10:
        k = best[1]
        for part, col in (('head', 'c'), ('tail', 'm'), ('chin', 'w')):
            p = keypoint(r, part, k); ax.plot(p[1], p[0], 'o', color=col, mec='k', ms=7, label=f'tracked {part}')
    ax.set_title(f'frame {fr}, mask fish {f}', fontsize=9); ax.legend(fontsize=6)

flips = rm[~rm.orient_ok]
display(flips[['frame', 'fish', 'csv_fish', 'n_low', 'n_high', 'n_components', 'tail_end_err', 'head_end_err']])
if len(flips):
    nc = min(len(flips), 4); nr = int(np.ceil(min(len(flips), 12) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.5 * nc, 4.2 * nr), squeeze=False)
    for ax, (_, row) in zip(axs.flat, flips.head(12).iterrows()):
        show_real(ax, row.frame, row.fish)
    for ax in axs.flat[len(flips):]: ax.axis('off')
    plt.tight_layout(); plt.show()

How close to failing are the others? The orientation rule compares the pixel counts of the two
halves. If flips only happened at ratios near 1 (near-ties), a tie-break rule could fix them.
If flipped masks have clearly unequal halves, the "head is bulkier" assumption itself is
wrong for those masks.

In [ ]:
ratio = np.maximum(rm.n_low, rm.n_high) / np.minimum(rm.n_low, rm.n_high)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(ratio[rm.orient_ok], bins=40, alpha=.7, label='correct')
ax.hist(ratio[~rm.orient_ok], bins=40, alpha=.9, color='r', label='flipped')
ax.set_xlabel('bigger half / smaller half (pixel count)'); ax.set_ylabel('masks'); ax.legend(); plt.show()
print(f'masks with ratio < 1.1 (fragile decisions): {(ratio < 1.1).sum()} of {len(rm)}')

### 7b. End-point distances to the tracked keypoints

Only orientation-correct masks are included (a flipped fit would count the length of the fish).
Large tail distances usually mean the mask misses the thin tail filament, so the sink sits inside the
true tail.

In [ ]:
ok = rm[rm.orient_ok]
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col, kp in zip(axs, ('tail_end_err', 'head_end_err'), ('tail', 'chin')):
    ax.hist(ok[col].dropna(), bins=40); ax.set_xlabel(f'fit end → tracked {kp} (px)')
    ax.set_title(f"median {ok[col].median():.1f}, 95th pct {ok[col].quantile(.95):.1f}, max {ok[col].max():.1f}")
plt.tight_layout(); plt.show()

worst = ok.sort_values('tail_end_err', ascending=False).head(4)
fig, axs = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (_, row) in zip(axs, worst.iterrows()):
    show_real(ax, row.frame, row.fish)
    ax.set_title(f'frame {row.frame}, fish {row.fish}: tail end {row.tail_end_err:.1f}px', fontsize=9)
plt.tight_layout(); plt.show()

### 7c. Mask contamination

Masks with more than one connected component, and masks with an unusually large pixel count for
their fish (attached blobs such as frame 0 fish 0 stay one component, so they are caught by size
instead). Both pull the PCA axis and the spline.

In [ ]:
med = rm.groupby('fish').n_pix.transform('median')
rm_ = rm.assign(size_ratio=rm.n_pix / med)
sus = rm_[(rm_.n_components > 1) | (rm_.size_ratio > 1.3)].sort_values('size_ratio', ascending=False)
print(f'{len(sus)} suspicious masks ({(rm_.n_components > 1).sum()} multi-component, '
      f'{(rm_.size_ratio > 1.3).sum()} > 1.3x the median size for that fish)')
display(sus[['frame', 'fish', 'n_pix', 'size_ratio', 'n_components', 'orient_ok', 'tail_end_err']].head(10).round(2))
if len(sus):
    fig, axs = plt.subplots(1, min(4, len(sus)), figsize=(4.5 * min(4, len(sus)), 4.5), squeeze=False)
    for ax, (_, row) in zip(axs.flat, sus.head(4).iterrows()):
        show_real(ax, row.frame, row.fish)
    plt.tight_layout(); plt.show()

### 7d. Real masks where the default spline overfits

In the synthetic tests the default always gave 2 knots (a single cubic). On real masks the
condition `residual ≤ s` sometimes fails for a cubic (fins, blobs, odd shapes increase the
scatter), so the spline adds knots. Because it is fitting the raw pixel cloud, that can jump
straight to hundreds of knots and a midline that zig-zags through edge pixels, the same failure
as in §5. `turning_deg` (total absolute turning along the curve) flags these.

In [ ]:
print('knot-count distribution:', rm.n_knots.value_counts().sort_index().to_dict())
over = rm[rm.n_knots > 2].sort_values('n_knots', ascending=False)
print(f'{len(over)} of {len(rm)} masks have interior knots; {(rm.n_knots > 50).sum()} have > 50')
display(over[['frame', 'fish', 'n_knots', 'turning_deg', 'n_components', 'orient_ok', 'tail_end_err']].head(15).round(1))
if len(over):
    nshow = min(4, len(over))
    fig, axs = plt.subplots(1, nshow, figsize=(4.5 * nshow, 4.5), squeeze=False)
    for ax, (_, row) in zip(axs.flat, over.head(nshow).iterrows()):
        show_real(ax, row.frame, row.fish)
        ax.set_title(f'frame {row.frame}, fish {row.fish}: {row.n_knots} knots, turning {row.turning_deg:.0f}°', fontsize=9)
    plt.tight_layout(); plt.show()

### 7e. Inspector

Look at any frame and fish: the mask with the fit and keypoints, and the `(t, w)` pixel cloud with
the spline and the halves used for the orientation decision.

In [ ]:
def inspect(fr, f):
    m = np.asarray(M[f, fr]); I = midline_internals(m)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[1, 1.6]))
    show_real(axs[0], fr, f)
    t_mid = 0.5 * (I['t'].min() + I['t'].max())
    axs[1].plot(I['t'], I['w'], ',', color='0.5')
    tt = np.linspace(I['t'].min(), I['t'].max(), 300); axs[1].plot(tt, I['spl'](tt), 'r')
    axs[1].axvline(t_mid, color='k', ls=':')
    axs[1].set_title(f"knots={I['n_knots']}, pixels low half={I['n_low']}, high half={I['n_high']} "
                     f"-> head at {'low' if I['flipped'] else 'high'} t", fontsize=9)
    axs[1].set_xlabel('t (px)'); axs[1].set_ylabel('w (px)'); axs[1].set_aspect('equal')
    plt.tight_layout(); plt.show()

inspect(0, 0)       # change frame / fish here

## 8. Candidate fix (comparison only; not used by the pipeline)

`fish_midline_v2` changes three things:
1. keeps only the **largest connected component** of the mask;
2. fits the spline to **1-px slice centroids** with an explicit pixel tolerance
   (`s = n_bins·sigma_px²`, default `sigma_px = 3`, chosen by the sweep at the end of this
   section) instead of the raw pixel cloud;
3. orients with the tracked **head/tail keypoints** when given, falling back to the
   bulkier-half rule otherwise.

Attached fins/blobs that are part of the same component (frame 0 fish 0) and very strong bends
(where `w(t)` is not single-valued) are **not** fixed by this.

In [ ]:
def fish_midline_v2(mask, n_ctrl=200, sigma_px=3.0, head=None, tail=None):
    '''(P,P) bool -> midline (n_ctrl,2) in [0,1], tail->head. head/tail: optional (row, col) px.'''
    lab, n = label(mask)
    if n > 1:
        mask = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    xs, ys = np.nonzero(mask)
    pts = np.column_stack([xs, ys]).astype(float); c = pts.mean(0)
    v = np.linalg.svd(pts - c, full_matrices=False)[2][0]
    perp = np.array([-v[1], v[0]])
    t = (pts - c) @ v; w = (pts - c) @ perp
    b = np.floor(t - t.min()).astype(int); cnt = np.bincount(b); k = cnt > 0
    tb = np.bincount(b, t)[k] / cnt[k]; wb = np.bincount(b, w)[k] / cnt[k]
    spl = UnivariateSpline(tb, wb, k=3, s=len(tb) * sigma_px ** 2)
    ts = np.linspace(t.min(), t.max(), n_ctrl)
    curve = c + np.outer(ts, v) + np.outer(spl(ts), perp)
    if head is not None and tail is not None and not (np.isnan(head).any() or np.isnan(tail).any()):
        flip = (np.linalg.norm(curve[0] - head) + np.linalg.norm(curve[-1] - tail)
                < np.linalg.norm(curve[-1] - head) + np.linalg.norm(curve[0] - tail))
    else:
        t_mid = 0.5 * (t.min() + t.max())
        flip = np.count_nonzero(t < t_mid) > np.count_nonzero(t >= t_mid)
    if flip: curve = curve[::-1]
    nx, ny = mask.shape
    return curve / np.array([nx - 1, ny - 1])

In [ ]:
rows = []
for kind, bend in SHAPES:
    for theta in (0, 17, 30, 45, 90):
        mid, mask = make_fish(theta, bend, kind)
        for name, cur in (('current', fish_midline_from_mask(mask)), ('v2', fish_midline_v2(mask))):
            rows.append(dict(shape=f'{kind}{bend}', rot=theta, version=name, **score(to_px(cur, PS), mid)))
cmp_s = pd.DataFrame(rows)
display(cmp_s.groupby(['shape', 'version'], sort=False)[['mean_err', 'max_err', 'tail_end_err', 'head_end_err']]
        .mean().unstack('version').round(2))
print('orientation correct (synthetic, no keypoints given):', cmp_s.groupby('version').orient_ok.mean().to_dict())

In [ ]:
rows = []
for _, row in rm.iterrows():
    m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
    h, t, ch = keypoint(r, 'head', k), keypoint(r, 'tail', k), keypoint(r, 'chin', k)
    cur = to_px(fish_midline_v2(m, head=h, tail=t), P)
    rows.append(dict(frame=row.frame, fish=row.fish, turning_deg=turning_deg(cur), **end_metrics(cur, h, t, ch)))
v2 = pd.DataFrame(rows)
print(f'curves turning > 360 deg (zig-zag): v2 {(v2.turning_deg > 360).sum()}, current {(rm.turning_deg > 360).sum()}  of {len(rm)}')
print(f'v2 orientation correct: {v2.orient_ok.sum()} / {len(v2)}  (current: {rm.orient_ok.sum()} / {len(rm)})')
print(f'v2 tail end -> tracked tail: median {v2.tail_end_err.median():.1f}px, 95th {v2.tail_end_err.quantile(.95):.1f}px')

if len(flips):
    fig, axs = plt.subplots(1, min(4, len(flips)), figsize=(4.5 * min(4, len(flips)), 4.5), squeeze=False)
    for ax, (_, row) in zip(axs.flat, flips.head(4).iterrows()):
        m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
        show_real(ax, row.frame, row.fish, curves=[
            (to_px(fish_midline_from_mask(m), P), 'y-', 'current'),
            (to_px(fish_midline_v2(m, head=keypoint(r, 'head', k), tail=keypoint(r, 'tail', k)), P), 'r-', 'v2')])
    plt.tight_layout(); plt.show()

### 8b. Choosing `sigma_px`: synthetic accuracy vs real-mask stability

Small `sigma_px` follows the synthetic midlines more closely, but the real masks (fins, ragged
edges) have much noisier cross-section centroids, and a tight tolerance then makes the spline
zig-zag. The sweep reports, per `sigma_px`, the worst synthetic error per shape (rotations 0°
and 30°) and, on the real masks, how many curves turn more than 180°/360° in total
(the current function's numbers are printed last for reference).

In [ ]:
rows = []
for sig in (1, 2, 3, 4, 6):
    rec = {'sigma_px': sig}
    for kind, bend in SHAPES:
        rec[f'max_err {kind}{bend}'] = max(score(to_px(fish_midline_v2(make_fish(th, bend, kind)[1], sigma_px=sig), PS),
                                                 make_fish(th, bend, kind)[0])['max_err'] for th in (0, 30))
    tr = np.array([turning_deg(to_px(fish_midline_v2(np.asarray(M[row.fish, row.frame]), sigma_px=sig), P))
                   for _, row in rm.iterrows()])
    rec.update({'real turning>180': (tr > 180).sum(), 'real turning>360': (tr > 360).sum(), 'real median turning': np.median(tr)})
    rows.append(rec)
display(pd.DataFrame(rows).round(1))
print(f"current function on real masks: turning>180: {(rm.turning_deg > 180).sum()}, "
      f">360: {(rm.turning_deg > 360).sum()}, median {rm.turning_deg.median():.0f}")

## 9. Alternative: tracing midline (`fish_midline_trace`)

The PCA method describes the midline as a sideways offset from one straight axis, which cannot
follow a fish that curls back on itself. The tracing method has no global axis:

1. **Trace**: start at the head point and walk `step` px at a time. Each new point is shifted
   sideways until the cross-section perpendicular to the walking direction is split into two equal
   halves (edges found to sub-pixel precision where the bilinear mask crosses 0.5, which is the pixel border).
   The direction is taken over the last `n_avg` steps. The walk stops when the cross-section is
   leaves the body; the tail tip is then the mask edge straight ahead. The points are reversed to run
   tail → head.
2. **Spline**: least-squares cubic spline `row(u), col(u)` through the traced points
   (`u` = length along them), with `n_knots` interior knots and both ends pinned.
3. **Recentre** (`n_refine` passes): redo the cross-sections perpendicular to the *spline*
   (smoother direction than the walk), move points to the middle, refit.

The head point must be given (synthetic: the true snout tip; real: the tracked `chin` keypoint,
snapped onto the mask if it falls just outside). The walk starts at the head because real tails are
thin and ragged, and the tracked `tail` keypoint often sits beside the tail rather than at its tip;
starting there, the walk set off sideways and left the body after 1–3 steps on 18 of 580 real masks.
Orientation therefore comes from the tracking, not from the mask.
The output format is identical to `fish_midline_from_mask`: `(n_ctrl, 2)` in [0, 1], tail → head,
evenly spaced by arc length.

In [ ]:
from scipy.ndimage import map_coordinates
from scipy.interpolate import make_lsq_spline


def _edges(F, P, N, half=40.0, res=0.25):
    '''Signed distances (lo < 0 < hi) from points P (k,2) along unit normals N (k,2) to the mask edge,
       staying in the inside run that contains each point. Sub-pixel: the edge is where the bilinear
       mask crosses 0.5 (= the pixel border). NaN where the point itself is outside the mask.'''
    o = np.arange(-half, half + res / 2, res); c = len(o) // 2; j = np.arange(len(o)); k = np.arange(len(P))
    xy = P[:, None, :] + o[None, :, None] * N[:, None, :]
    V = map_coordinates(F, xy.reshape(-1, 2).T, order=1, mode='constant').reshape(len(P), len(o))
    out = V < 0.5
    jl = np.where(out & (j < c), j, -1).max(1)            # last outside sample left of the point
    jr = np.where(out & (j > c), j, len(o)).min(1)        # first outside sample right of the point

    def cross(jo, ji):                                   # linear interpolation of V across 0.5
        joc = np.clip(jo, 0, len(o) - 1)
        vo, vi = V[k, joc], V[k, ji]
        with np.errstate(divide='ignore', invalid='ignore'):   # vi == vo only where jo is masked below
            e = o[joc] + (0.5 - vo) / (vi - vo) * (o[ji] - o[joc])
        return np.where((jo < 0) | (jo >= len(o)), o[ji], e)

    lo, hi = cross(jl, np.clip(jl + 1, 0, c)), cross(jr, np.clip(jr - 1, c, len(o) - 1))
    lo[out[:, c]] = np.nan; hi[out[:, c]] = np.nan
    return lo, hi


def _resample(pts, n):
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pts, axis=0), axis=1))]
    u = np.linspace(0, s[-1], n)
    return np.column_stack([np.interp(u, s, pts[:, 0]), np.interp(u, s, pts[:, 1])])


def trace_midline(F, start, step=3.0, n_avg=2, max_steps=500):
    '''Walk from `start` (row, col, inside the mask) to the other tip. Each new point is placed `step`
       px ahead and moved sideways until the cross-section perpendicular to the walking direction is
       split in two equal halves. Returns the traced points (m,2) in px, start first.'''
    pix = np.argwhere(F > 0.5)
    d = pix[np.linalg.norm(pix - start, axis=1) < 4 * step].mean(0) - start   # into the body
    d /= np.linalg.norm(d)
    pts = [np.asarray(start, float)]
    for _ in range(max_steps):
        nrm = np.array([-d[1], d[0]])
        q = pts[-1] + step * d
        for _ in range(2):                                   # centre q on its cross-section
            lo, hi = _edges(F, q[None], nrm[None])
            if np.isnan(lo[0]): break
            q = q + 0.5 * (lo[0] + hi[0]) * nrm
        if np.isnan(lo[0]):                                  # left the body: reached the far tip
            break
        pts.append(q)
        back = pts[-1 - min(n_avg, len(pts) - 1)]
        d = (q - back) / np.linalg.norm(q - back)
    _, hi = _edges(F, pts[-1][None], d[None])                 # far tip: straight on to the edge
    pts.append(pts[-1] + hi[0] * d)
    return np.array(pts)


def fit_spline(pts, n_knots=10, n_out=200):
    '''Least-squares cubic spline row(u), col(u) through pts (u = cumulative length), ends pinned,
       returned as n_out points evenly spaced by arc length.'''
    u = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pts, axis=0), axis=1))]
    keep = np.r_[True, np.diff(u) > 1e-9]; pts, u = pts[keep], u[keep] / u[keep][-1]
    k = min(3, len(pts) - 1)                                     # very short traces: lower degree
    n_int = int(np.clip(len(pts) // 4 - 1, 0, n_knots))
    t = np.r_[[0.0] * (k + 1), np.linspace(0, 1, n_int + 2)[1:-1], [1.0] * (k + 1)]
    w = np.ones(len(pts)); w[[0, -1]] = 1e3
    spl = make_lsq_spline(u, pts, t, k=k, w=w)
    return _resample(spl(np.linspace(0, 1, 20 * n_out)), n_out)


def recentre(F, curve):
    '''Move each interior point to the middle of the cross-section perpendicular to the curve.'''
    tan = np.gradient(curve, axis=0); tan /= np.linalg.norm(tan, axis=1, keepdims=True)
    nrm = np.column_stack([-tan[:, 1], tan[:, 0]])
    lo, hi = _edges(F, curve[1:-1], nrm[1:-1])
    new = curve.copy(); new[1:-1] += np.nan_to_num(0.5 * (lo + hi))[:, None] * nrm[1:-1]
    return new


def fish_midline_trace(mask, head, step=3.0, n_knots=10, n_refine=1, n_ctrl=200):
    '''(P,P) bool mask + head point (row, col) in px, e.g. the tracked chin -> midline (n_ctrl,2) in
       [0,1], tail->head, evenly spaced by arc length (same output format as fish_midline_from_mask).
       The walk starts at the head (wide, well tracked) and ends where it leaves the body at the tail.'''
    F = mask.astype(float)
    head = np.asarray(head, float)
    if map_coordinates(F, head[:, None], order=1)[0] < 0.5:   # keypoint just off the mask: snap
        pix = np.argwhere(mask); head = pix[np.argmin(np.linalg.norm(pix - head, axis=1))].astype(float)
    pts = trace_midline(F, head, step)[::-1]
    m = len(pts)
    curve = fit_spline(pts, n_knots, m)
    for _ in range(n_refine):
        pts = recentre(F, curve)
        curve = fit_spline(pts, n_knots, m)
    curve = fit_spline(pts, n_knots, n_ctrl)
    return curve / (np.array(mask.shape) - 1)

In [ ]:
mid, mask = make_fish(30, 270, 'C')
F = mask.astype(float)
pts = trace_midline(F, mid[-1])
cur = to_px(fish_midline_trace(mask, mid[-1]), PS)
fig, ax = plt.subplots(figsize=(7, 6))
show_fit(ax, mask, [(cur, 'r-', 'trace + spline')], truth=mid, title='C-bend 270°: traced points and final midline')
d = np.gradient(pts, axis=0); d /= np.linalg.norm(d, axis=1, keepdims=True); nrm = np.column_stack([-d[:, 1], d[:, 0]])
lo, hi = _edges(F, pts[1:-1], nrm[1:-1])
for p, n_, a, b in zip(pts[1:-1], nrm[1:-1], lo, hi):            # the cross-sections the walk centred on
    ax.plot([p[1] + a * n_[1], p[1] + b * n_[1]], [p[0] + a * n_[0], p[0] + b * n_[0]], 'c-', lw=0.6)
ax.plot(pts[:, 1], pts[:, 0], 'c.', ms=4, label='traced points')
ax.legend(fontsize=7); plt.show()
print(f'{len(pts)} traced points; error vs truth: ' + ', '.join(f'{k} {v:.2f}' for k, v in score(cur, mid).items() if k != 'orient_ok'))

## 10. PCA vs tracing on synthetic fish

Same synthetic fish as before plus stronger bends (C200–C320, S120, S180), 5 rotations each.
`ms` is the run time per fish. The trace starts at the true snout, so its `head_end_err` is 0 by
construction; §10b checks what happens when the start point is off.

In [ ]:
import time
SHAPES_ALL = SHAPES + [('C', 200), ('C', 270), ('C', 320), ('S', 120), ('S', 180)]

def timed(f, *a, **kw):
    t0 = time.perf_counter(); out = f(*a, **kw); return out, 1e3 * (time.perf_counter() - t0)

rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 17, 30, 45, 90):
        mid, mask = make_fish(theta, bend, kind)
        for name, f, a in (('PCA', fish_midline_from_mask, ()), ('trace', fish_midline_trace, (mid[-1],))):
            cur, ms = timed(f, mask, *a)
            rows.append(dict(shape=f'{kind}{bend}', rot=theta, method=name, ms=ms, **score(to_px(cur, PS), mid)))
cmp_syn = pd.DataFrame(rows)
display(cmp_syn.groupby(['shape', 'method'], sort=False)[['mean_err', 'max_err', 'tail_end_err', 'head_end_err', 'ms']]
        .mean().unstack('method').round(2))

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for ax, (kind, bend) in zip(axs.flat, [('C', 150), ('C', 200), ('C', 270), ('C', 320), ('S', 120), ('S', 180)]):
    mid, mask = make_fish(30, bend, kind)
    show_fit(ax, mask, [(to_px(fish_midline_from_mask(mask), PS), 'y-', 'PCA'),
                        (to_px(fish_midline_trace(mask, mid[-1]), PS), 'r-', 'trace')],
             truth=mid, title=f'{kind}-bend {bend}°')
plt.tight_layout(); plt.show()

### 10b. Start point and parameters

On real data the head point comes from tracking and won't be exact. Below, the trace starts 3 px
away from the true snout in four directions (snapped onto the mask if that lands outside). Then
`step` and `n_refine` are varied (all shapes, rotation 30°).

In [ ]:
rows = []
for kind, bend in SHAPES_ALL:
    mid, mask = make_fish(30, bend, kind)
    for off in ((0, 0), (3, 0), (-3, 0), (0, 3), (0, -3)):
        cur = to_px(fish_midline_trace(mask, mid[-1] + np.array(off)), PS)
        rows.append(dict(shape=f'{kind}{bend}', offset=str(off), **score(cur, mid)))
off_df = pd.DataFrame(rows)
display(off_df.groupby('offset', sort=False)[['mean_err', 'max_err', 'tail_end_err', 'head_end_err']].max().round(2)
        .rename(columns=lambda c: 'worst ' + c))

rows = []
for step in (2, 3, 5):
    for n_refine in (0, 1, 2):
        for kind, bend in SHAPES_ALL:
            mid, mask = make_fish(30, bend, kind)
            cur, ms = timed(fish_midline_trace, mask, mid[-1], step=step, n_refine=n_refine)
            rows.append(dict(step=step, n_refine=n_refine, ms=ms, **score(to_px(cur, PS), mid)))
display(pd.DataFrame(rows).groupby(['step', 'n_refine']).agg(mean_err=('mean_err', 'mean'), worst_max_err=('max_err', 'max'),
                                                              ms=('ms', 'mean')).round(2))

## 11. PCA vs tracing on real masks

The same matched masks as §7. The trace starts from the tracked `chin` keypoint of the matched
CSV fish, so its orientation is correct by construction and its head end is the (snapped) chin.
There is no true midline here, so we compare:

* `tail_end_err`: midline's tail end → tracked `tail` (independent for both methods: the trace finds
  its tail end itself, where the walk leaves the body);
* `head_end_err`: midline's head end → tracked `chin` (only meaningful for PCA; ≈0 for the trace by construction);
* `inside`: fraction of the 200 midline points that lie inside the mask (a midline should never leave the body);
* `turning_deg`: total turning (zig-zag detector, as in §7d);
* `disagreement`: mean distance from the trace to the PCA curve, to find where the two differ;
* `ms`: run time per fish.

In [ ]:
def inside_frac(mask, cur):
    return (map_coordinates(mask.astype(float), cur.T, order=1) >= 0.5).mean()

rows = []
for _, row in rm.iterrows():
    m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
    chin, tail = keypoint(r, 'chin', k), keypoint(r, 'tail', k)
    c_pca, ms_pca = timed(fish_midline_from_mask, m)
    c_tr, ms_tr = timed(fish_midline_trace, m, chin)
    c_pca, c_tr = to_px(c_pca, P), to_px(c_tr, P)
    rec = dict(frame=row.frame, fish=row.fish, csv_fish=k, disagreement=dist_to_poly(c_tr, c_pca).mean())
    for name, c, ms in (('PCA', c_pca, ms_pca), ('trace', c_tr, ms_tr)):
        rec.update({f'tail_end_err {name}': np.linalg.norm(c[0] - tail), f'head_end_err {name}': np.linalg.norm(c[-1] - chin),
                    f'length {name}': np.linalg.norm(np.diff(c, axis=0), axis=1).sum(), f'inside {name}': inside_frac(m, c),
                    f'turning_deg {name}': turning_deg(c), f'ms {name}': ms})
    rows.append(rec)
cmp_real = pd.DataFrame(rows)

summ = {}
for name in ('PCA', 'trace'):
    summ[name] = {'tail end → tracked tail: median px': cmp_real[f'tail_end_err {name}'].median(),
                  'tail end → tracked tail: 95th pct px': cmp_real[f'tail_end_err {name}'].quantile(.95),
                  'tail end → tracked tail: > 20 px (count)': (cmp_real[f'tail_end_err {name}'] > 20).sum(),
                  'head end → chin: median px': cmp_real[f'head_end_err {name}'].median(),
                  'head end → chin: 95th pct px': cmp_real[f'head_end_err {name}'].quantile(.95),
                  'head end → chin: > 20 px (count)': (cmp_real[f'head_end_err {name}'] > 20).sum(),
                  'midlines with < 95% of points inside the mask': (cmp_real[f'inside {name}'] < .95).sum(),
                  'zig-zag (turning > 360°)': (cmp_real[f'turning_deg {name}'] > 360).sum(),
                  'median turning °': cmp_real[f'turning_deg {name}'].median(),
                  'ms per fish (median)': cmp_real[f'ms {name}'].median(),
                  'ms per fish (max)': cmp_real[f'ms {name}'].max()}
print(f'{len(cmp_real)} real masks')
display(pd.DataFrame(summ).round(2))

### 11a. Side by side

Yellow = PCA, red = trace (triangles mark each method's head end); cyan/magenta/white = tracked
head/tail/chin. Rows: (1) the cases PCA got wrong in §7 (flipped, then zig-zag), (2) the masks where
the two methods disagree most, (3) the trace's worst tail-end errors, where the trace itself has
failed.

In [ ]:
def compare_row(cases, title):
    cases = list(cases)[:4]
    fig, axs = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, (fr, f) in zip(axs, cases):
        m = np.asarray(M[f, fr]); r = df.loc[fr + 1]
        k = cmp_real.set_index(['frame', 'fish']).loc[(fr, f), 'csv_fish']
        show_real(ax, fr, f, curves=[(to_px(fish_midline_from_mask(m), P), 'y-', 'PCA'),
                                     (to_px(fish_midline_trace(m, keypoint(r, 'chin', k)), P), 'r-', 'trace')])
    for ax in axs[len(cases):]: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

pairs = lambda d: zip(d.frame, d.fish)
compare_row(list(pairs(flips)) + list(pairs(over)), '(1) PCA failures from §7: flipped, then zig-zag')
compare_row(pairs(cmp_real.sort_values('disagreement', ascending=False)), '(2) largest disagreement between methods')
compare_row(pairs(cmp_real.sort_values('tail_end_err trace', ascending=False)), '(3) worst trace tail-end errors')
compare_row(pairs(cmp_real.sort_values('tail_end_err trace', ascending=False).iloc[4:]), '(3, cont.) next worst trace tail-end errors')

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, col, xl in zip(axs, ('tail_end_err', 'turning_deg', 'inside'), ('tail end → tracked tail (px)', 'total turning (°)', 'fraction of midline inside mask')):
    for name, colr in (('PCA', 'goldenrod'), ('trace', 'r')):
        v = cmp_real[f'{col} {name}'].clip(upper={'tail_end_err': 60, 'turning_deg': 720, 'inside': 1}[col])
        ax.hist(v, bins=40, alpha=.55, color=colr, label=name)
    ax.set_xlabel(xl); ax.legend()
axs[0].set_title('clipped at 60 px', fontsize=8); axs[1].set_title('clipped at 720°', fontsize=8)
plt.tight_layout(); plt.show()

## 12. Three more methods: principal curve, Isomap, Denis (EODAssignment)

* **Principal curve** (`fish_midline_pcurve`, Hastie & Stuetzle 1989): the nonlinear version of PCA's
  first component. Starts from the PCA line and repeats: give every pixel the position `u` of its
  nearest curve point, refit `row(u), col(u)` to all pixels. The smoother is a P-spline (10 evenly
  spaced knots + a small bending penalty, `lam = 1e-3`). Two guards were needed: with a plain
  least-squares spline the fit became undefined once a knot interval had no pixels, and the snout
  end (few pixels) grew a hook outside the body that lengthened every pass. So each pass the curve
  is cut to the part pixels project onto and that lies inside the mask.
* **Isomap** (`fish_midline_isomap`, Tenenbaum et al. 2000; landmark version, de Silva & Tenenbaum
  2003, written with scipy since scikit-learn isn't installed): distances *through the body* along the
  8-connected pixel graph from 100 random landmark pixels, embedded in 1-D by classical MDS. The 1-D
  coordinate runs along the body however it is bent. The midline is the mean pixel position in 1-px bins of
  that coordinate, smoothed by a 10-knot spline with pinned ends.
* **Denis** (`fish_midline_denis`): a port of `estimate_point_currents_location` from
  [DenisTurcu/EODAssignment](https://github.com/DenisTurcu/EODAssignment) (`src/field_model.py`). It
  uses **no mask**, only the 4 tracked keypoints tail → middle → head → mouth (chin ignored). Each
  coordinate is interpolated over the uniform parameter `t = 0, 1/3, 2/3, 1` with a piecewise-cubic
  Hermite (Fritsch–Butland tangents: zero at both ends and at any keypoint where that coordinate
  turns), oversampled to `n × 100` points, and `n` points are picked evenly by arc length. The port was
  checked against the original functions (decorators removed) on 500 random keypoint sets: points
  1…n−1 agree to 6e-14. The last point has a quirk that the port reproduces: the original looks for
  an interval with `cum[j] ≤ 1 < cum[j+1]`; when rounding leaves the cumulative length ≤ 1, it
  falls back to "one oversampled step after the previous point". The last current then sits
  ~1/n of the body length short of the mouth. This happened in 264/500 keypoint sets for both the
  original and the port.

All mask methods are oriented with the head point (synthetic: true snout; real: tracked chin).
PCA keeps its own bulkier-half rule, as in the pipeline.

In [ ]:
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import cKDTree
from scipy.ndimage import label, map_coordinates
from scipy.interpolate import make_lsq_spline, BSpline


def _largest_component(mask):
    lab, n = label(mask, structure=np.ones((3, 3)))
    return mask if n <= 1 else lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)


def _lsq_curve(u, pts, n_knots, w=None):
    '''least-squares cubic spline row(u), col(u) (u in [0,1], sorted) with n_knots evenly spaced interior knots'''
    t = np.r_[[0.0] * 4, np.linspace(0, 1, n_knots + 2)[1:-1], [1.0] * 4]
    return make_lsq_spline(u, pts, t, k=3, w=w)


def _orient(curve, head):
    '''tail -> head: flip if the first point is closer to the head point'''
    return curve[::-1] if np.linalg.norm(curve[0] - head) < np.linalg.norm(curve[-1] - head) else curve


# ---------------------------------------------------------------- principal curve
def _pspline(u, y, n_knots, lam):
    '''penalised least-squares cubic spline (P-spline): evenly spaced knots + lam * squared second
       differences of the coefficients, so it stays well-posed where a knot interval has no data'''
    t = np.r_[[0.0] * 4, np.linspace(0, 1, n_knots + 2)[1:-1], [1.0] * 4]
    B = BSpline.design_matrix(u, t, 3).toarray()
    D = np.diff(np.eye(B.shape[1]), 2, axis=0)
    return BSpline(t, np.linalg.solve(B.T @ B + lam * len(u) * D.T @ D, B.T @ y), 3)


def fish_midline_pcurve(mask, head, n_knots=10, lam=1e-3, n_iter=30, tol=0.05, n_ctrl=200):
    '''Principal curve (Hastie & Stuetzle 1989) of the mask pixels with a P-spline smoother. Starts
       from the first principal component (a line), then repeats: project every pixel onto the
       nearest point of the curve (u = arc-length position), refit row(u), col(u) to all pixels.
       Each pass the curve is cut to the stretch that pixels project onto and that lies inside the
       mask (otherwise the end with few pixels, the snout, overshoots into a hook that grows every
       pass). Stops when the curve moves < tol px. Oriented with `head`.'''
    mask = _largest_component(mask); F = mask.astype(float)
    pix = np.argwhere(mask).astype(float)
    c = pix.mean(0); v = np.linalg.svd(pix - c, full_matrices=False)[2][0]
    t = (pix - c) @ v
    curve = c + np.outer(np.linspace(t.min(), t.max(), 1000), v)
    for _ in range(n_iter):
        s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(curve, axis=0), axis=1))]
        u = s[cKDTree(curve).query(pix)[1]]
        u = (u - u.min()) / (u.max() - u.min())
        new = _pspline(u, pix, n_knots, lam)(np.linspace(0, 1, 4000))
        ins = np.nonzero(map_coordinates(F, new.T, order=1) >= 0.5)[0]
        new = _resample(new[ins[0]:ins[-1] + 1], 1000)
        moved = np.linalg.norm(new - curve, axis=1).max()
        curve = new
        if moved < tol: break
    return _orient(_resample(curve, n_ctrl), head) / (np.array(mask.shape) - 1)


# ---------------------------------------------------------------- isomap
def _pixel_graph(mask):
    '''8-connected graph of mask pixels, edge weight = Euclidean step (1 or sqrt 2)'''
    idx = -np.ones(mask.shape, int); rr, cc = np.nonzero(mask); idx[rr, cc] = np.arange(len(rr))
    a, b, w = [], [], []
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
        r2, c2 = rr + dr, cc + dc
        ok = (r2 < mask.shape[0]) & (c2 >= 0) & (c2 < mask.shape[1])
        ok[ok] &= mask[r2[ok], c2[ok]]
        a.append(idx[rr[ok], cc[ok]]); b.append(idx[r2[ok], c2[ok]]); w.append(np.full(ok.sum(), np.hypot(dr, dc)))
    a, b, w = map(np.concatenate, (a, b, w))
    G = sp.csr_matrix((np.r_[w, w], (np.r_[a, b], np.r_[b, a])), shape=(len(rr), len(rr)))
    return G, np.column_stack([rr, cc]).astype(float)


def fish_midline_isomap(mask, head, n_landmarks=100, n_knots=10, n_ctrl=200, seed=0):
    '''Isomap (Tenenbaum et al. 2000, landmark version of de Silva & Tenenbaum 2003) of the mask
       pixels: distances measured through the body along the 8-connected pixel graph, embedded in
       1-D by classical MDS. The 1-D coordinate runs along the body however it is bent; the
       midline is the mean pixel position in 1-px bins of that coordinate, smoothed by a fixed-knot
       spline (ends pinned). Oriented with `head`.'''
    G, pix = _pixel_graph(_largest_component(mask))
    lm = np.random.default_rng(seed).choice(len(pix), min(n_landmarks, len(pix)), replace=False)
    D2 = dijkstra(G, indices=lm) ** 2                        # (m, n) squared geodesic distances
    Dl = D2[:, lm]
    m = len(lm); H = np.eye(m) - 1.0 / m
    lam, vec = np.linalg.eigh(-0.5 * H @ Dl @ H)
    y = -0.5 * (vec[:, -1] / np.sqrt(lam[-1])) @ (D2 - Dl.mean(1)[:, None])   # 1-D embedding of every pixel
    b = np.floor(y - y.min()).astype(int); cnt = np.bincount(b); k = cnt > 0
    pts = np.column_stack([np.bincount(b, pix[:, 0])[k], np.bincount(b, pix[:, 1])[k]]) / cnt[k][:, None]
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pts, axis=0), axis=1))]
    w = np.ones(len(pts)); w[[0, -1]] = 1e3
    spl = _lsq_curve(s / s[-1], pts, min(n_knots, len(pts) // 4), w)
    return _orient(_resample(spl(np.linspace(0, 1, 4000)), n_ctrl), head) / (np.array(mask.shape) - 1)


# ---------------------------------------------------------------- Denis (EODAssignment)
def _fb_interp(x, y, x_new):
    '''Vectorised port of EODAssignment src/field_model.py::_interpolate_points: piecewise cubic
       Hermite with Fritsch-Butland tangents (harmonic mean of neighbouring slopes, 0 at a local
       extremum and at the two ends), evaluated at x_new (constant outside [x0, x_end]).'''
    n = len(x); m = np.diff(y) / np.diff(x)                   # slope of each interval
    tan = np.zeros(n)
    with np.errstate(divide='ignore', invalid='ignore'):
        inner = 2 * m[:-1] * m[1:] / (m[:-1] + m[1:])
    tan[1:-1] = np.where(m[:-1] * m[1:] > 0, inner, 0.0)
    i = np.clip(np.searchsorted(x, x_new, side='right') - 1, 0, n - 2)
    h = x[i + 1] - x[i]; t = (x_new - x[i]) / h
    out = ((2 * t**3 - 3 * t**2 + 1) * y[i] + (t**3 - 2 * t**2 + t) * h * tan[i]
           + (-2 * t**3 + 3 * t**2) * y[i + 1] + (t**3 - t**2) * h * tan[i + 1])
    return np.where(x_new <= x[0], y[0], np.where(x_new >= x[-1], y[-1], out))


def fish_midline_denis(keypoints, shape, n_ctrl=200, uniformity_multiplier=100):
    '''Port of EODAssignment estimate_point_currents_location for one fish: keypoints (4,2) =
       tail, middle, head, mouth; interpolated per coordinate at uniform parameter t = 0, 1/3, 2/3, 1;
       oversampled to n_ctrl*uniformity_multiplier points; n_ctrl points picked evenly by arc length
       (each = the oversampled point just before the target length). No mask is used; `shape` is the
       mask shape, only for the [0,1] scaling. Returns (n_ctrl,2) in [0,1], tail -> mouth.'''
    kp = np.asarray(keypoints, float); t = np.linspace(0, 1, len(kp))
    ts = np.linspace(0, 1, n_ctrl * uniformity_multiplier)
    over = np.column_stack([_fb_interp(t, kp[:, d], ts) for d in range(2)])
    dx, dy = np.diff(over[:, 0]), np.diff(over[:, 1])
    seg = np.sqrt(dx * dx + dy * dy)                          # same arithmetic as the original, so that
    cum = np.r_[0, np.cumsum(seg / seg.sum())]                # floating-point rounding matches too
    idx = np.searchsorted(cum, np.linspace(0, 1, n_ctrl), side='right') - 1
    idx[idx >= len(ts) - 1] = 0                               # original: no interval found -> 0 ...
    for k in range(1, n_ctrl):                                # ... then forced strictly increasing
        if idx[k] <= idx[k - 1]: idx[k] = idx[k - 1] + 1
    # quirk kept from the original: if rounding leaves cum[-1] <= 1, the last point is placed one
    # oversampled step after the previous one (~1/n_ctrl of the length short of the mouth)
    return over[idx] / (np.array(shape) - 1)


def _resample(pts, n):
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pts, axis=0), axis=1))]
    u = np.linspace(0, s[-1], n)
    return np.column_stack([np.interp(u, s, pts[:, 0]), np.interp(u, s, pts[:, 1])])

## 13. Five methods on synthetic fish

Denis's method needs keypoints. Here they are taken from the **true** midline at the median arc-length
positions of the tracked keypoints in the real data (tail = 0, middle, head, mouth = snout). That makes
this the best case for that method: no tracking noise. Errors are in px vs the true midline; `ms` is the time per fish.

In [ ]:
_frac = []
for k in range(1, 5):
    _P = np.stack([df[[f'{p}_y_{k}', f'{p}_x_{k}']].to_numpy() for p in ('tail', 'middle', 'head', 'mouth')], 1)
    _seg = np.linalg.norm(np.diff(_P, axis=1), axis=2); _ok = ~np.isnan(_seg).any(1)
    _frac.append(np.cumsum(_seg[_ok], 1) / _seg[_ok].sum(1, keepdims=True))
KP_FRAC = np.median(np.vstack(_frac), 0)
print('median arc-length position of middle, head, mouth along tail→mouth:', KP_FRAC.round(3))

def synth_keypoints(mid):
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(mid, axis=0), axis=1))]; s /= s[-1]
    return np.array([mid[0], mid[np.searchsorted(s, KP_FRAC[0])], mid[np.searchsorted(s, KP_FRAC[1])], mid[-1]])

METHODS = {'PCA':    lambda m, kp, head: fish_midline_from_mask(m),
           'trace':  lambda m, kp, head: fish_midline_trace(m, head),
           'pcurve': lambda m, kp, head: fish_midline_pcurve(m, head),
           'isomap': lambda m, kp, head: fish_midline_isomap(m, head),
           'denis':  lambda m, kp, head: fish_midline_denis(kp, m.shape)}
STYLE = {'PCA': 'y-', 'trace': 'r-', 'pcurve': 'b-', 'isomap': 'g-', 'denis': 'm-'}

rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, mask = make_fish(theta, bend, kind); kp = synth_keypoints(mid)
        for name, f in METHODS.items():
            cur, ms = timed(f, mask, kp, mid[-1])
            rows.append(dict(shape=f'{kind}{bend}', rot=theta, method=name, ms=ms, **score(to_px(cur, PS), mid)))
five_syn = pd.DataFrame(rows)
for col in ('mean_err', 'max_err'):
    print(col); display(five_syn.pivot_table(index='shape', columns='method', values=col, sort=False)[list(METHODS)].round(2))
print('ms per fish'); display(five_syn.groupby('method').ms.median().loc[list(METHODS)].round(1).to_frame().T)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 10))
for ax, (kind, bend) in zip(axs.flat, [('C', 90), ('C', 150), ('C', 270), ('C', 320), ('S', 120), ('S', 180)]):
    mid, mask = make_fish(30, bend, kind); kp = synth_keypoints(mid)
    show_fit(ax, mask, [(to_px(f(mask, kp, mid[-1]), PS), STYLE[n], n) for n, f in METHODS.items()],
             truth=mid, title=f'{kind}-bend {bend}°')
    ax.plot(kp[:, 1], kp[:, 0], 'mo', mec='k', ms=6)
plt.tight_layout(); plt.show()

## 14. Five methods on real masks

Same 580 matched masks as §7. Mask methods use the tracked `chin` as head point; Denis uses the
tracked tail, middle, head and mouth keypoints. There is no true midline, so the checks are:

* `tail_end_err`: tail end → tracked `tail` (0 for Denis by construction);
* `head_end_err`: head end → tracked `chin` (≈0 for the trace by construction; Denis ends at `mouth`);
* `inside`: fraction of midline points inside the mask (Denis never sees the mask, so this tests
  whether the keypoint curve stays in the body);
* `turning_deg`: total turning (zig-zag detector);
* pairwise distances between methods: where they agree, and which method is the odd one out.

In [ ]:
rows, curves_real = [], {}
for _, row in rm.iterrows():
    m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
    chin, tail = keypoint(r, 'chin', k), keypoint(r, 'tail', k)
    kp = np.array([keypoint(r, p, k) for p in ('tail', 'middle', 'head', 'mouth')])
    cs = {}
    for name, f in METHODS.items():
        if name == 'denis' and np.isnan(kp).any(): continue
        cur, ms = timed(f, m, kp, chin); cur = to_px(cur, P); cs[name] = cur
        rows.append(dict(frame=row.frame, fish=row.fish, method=name, ms=ms, tail_end_err=np.linalg.norm(cur[0] - tail),
                         head_end_err=np.linalg.norm(cur[-1] - chin), inside=inside_frac(m, cur), turning_deg=turning_deg(cur),
                         length=np.linalg.norm(np.diff(cur, axis=0), axis=1).sum()))
    curves_real[(row.frame, row.fish)] = cs
five_real = pd.DataFrame(rows)
g = five_real.groupby('method')
summary = pd.DataFrame({
    'tail end → tail: median px': g.tail_end_err.median(), 'tail end → tail: > 20 px': g.tail_end_err.apply(lambda v: (v > 20).sum()),
    'head end → chin: median px': g.head_end_err.median(), 'head end → chin: > 20 px': g.head_end_err.apply(lambda v: (v > 20).sum()),
    '< 95% inside mask': g.inside.apply(lambda v: (v < .95).sum()), 'zig-zag (> 360°)': g.turning_deg.apply(lambda v: (v > 360).sum()),
    'median turning °': g.turning_deg.median(), 'ms median': g.ms.median(), 'ms max': g.ms.max(), 'n masks': g.size()}).T
display(summary[list(METHODS)].round(1))

Pairwise agreement: for every mask, the mean distance (px) from one method's midline to the other's,
then the median over masks (lower triangle) and the number of masks where they differ by more than 5 px
on average (upper triangle).

In [ ]:
names = list(METHODS)
med = pd.DataFrame(np.nan, index=names, columns=names)
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i == j: continue
        d = np.array([dist_to_poly(cs[a], cs[b]).mean() for cs in curves_real.values() if a in cs and b in cs])
        med.loc[a, b] = np.median(d) if i > j else (d > 5).sum()
display(med.round(2))

In [ ]:
def five_row(keys, title):
    keys = list(keys)[:4]
    fig, axs = plt.subplots(1, 4, figsize=(19, 4.8))
    for ax, (fr, f) in zip(axs, keys):
        cs = curves_real[(fr, f)]
        show_real(ax, fr, f, curves=[(c, STYLE[n], n) for n, c in cs.items()])
    for ax in axs[len(keys):]: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

iso = five_real[five_real.method == 'isomap'].set_index(['frame', 'fish'])
spread = {key: np.max([dist_to_poly(cs[a], cs[b]).mean() for a in cs for b in cs if a != b and 'PCA' not in (a, b)])
          for key, cs in curves_real.items()}
five_row(list(zip(flips.frame, flips.fish)) + list(zip(over.frame, over.fish)), 'PCA failures from §7 (flipped, then zig-zag)')
five_row(iso.turning_deg.sort_values(ascending=False).index, 'most bent fish (by the Isomap midline)')
five_row(sorted(spread, key=spread.get, reverse=True), 'largest disagreement among trace / pcurve / isomap / denis')
five_row(sorted(spread, key=spread.get, reverse=True)[4:], '(cont.)')

## 15. Fin removal (width envelope)

`remove_fins` cuts the pectoral fins off a mask before any midline is fitted:

1. Midline of the mask (Isomap with only 6 knots, so the fins pull it less).
2. At every midline point, measure how far the body reaches on each side (perpendicular cut).
3. Median-filter those two side widths over 20 % of the body length. A fin is narrower than half
   that window, so it drops out as an outlier and leaves the smooth body width. Envelope =
   1.25 × that width + 1.5 px.
4. Drop every pixel that lies further sideways than the envelope from its nearest midline point
   (and more than 3 px past either end). Keep the largest connected piece.
5. Repeat once, with the midline now fitted on the cleaned mask but the envelope measured on the
   original mask.

Both pectoral fins are removed in one pass; small stubs within the 1.25× margin stay. It costs ~60 ms per fish
(two Isomap fits).

In [ ]:
from scipy.ndimage import median_filter, label
from scipy.spatial import cKDTree


def _frame(curve):
    tan = np.gradient(curve, axis=0); tan /= np.linalg.norm(tan, axis=1, keepdims=True)
    return tan, np.column_stack([-tan[:, 1], tan[:, 0]])


def body_envelope(mask, curve, win_frac=0.2, k=1.25, pad=1.5):
    '''Half-widths of the body on each side of `curve` (px, (n,2)): the measured extent along each normal,
       median-filtered over win_frac of the body length (fins are shorter than half the window, so they
       are filtered out as outliers), times k plus pad px. Returns (env_left, env_right, a, b) with
       a, b the raw measured extents.'''
    tan, nrm = _frame(curve)
    lo, hi = _edges(mask.astype(float), curve, nrm)
    a, b = -lo, hi
    ok = ~np.isnan(a); idx = np.arange(len(a))
    a, b = np.interp(idx, idx[ok], a[ok]), np.interp(idx, idx[ok], b[ok])   # points just outside the mask
    w = max(3, int(win_frac * len(curve)) | 1)
    return k * median_filter(a, w, mode='nearest') + pad, k * median_filter(b, w, mode='nearest') + pad, a, b


def clip_to_envelope(mask, curve, env_l, env_r, beyond=3.0):
    '''Keep mask pixels whose nearest midline point j has them within [-env_l[j], env_r[j]] sideways
       (and, past the two ends, at most `beyond` px further along); largest connected piece.'''
    tan, nrm = _frame(curve)
    pix = np.argwhere(mask).astype(float)
    j = cKDTree(curve).query(pix)[1]
    d = pix - curve[j]
    lat, along = (d * nrm[j]).sum(1), (d * tan[j]).sum(1)
    keep = (lat >= -env_l[j]) & (lat <= env_r[j])
    keep &= ~((j == 0) & (along < -beyond)) & ~((j == len(curve) - 1) & (along > beyond))
    out = np.zeros_like(mask); out[tuple(pix[keep].astype(int).T)] = True
    return _largest_component(out)


def remove_fins(mask, head, n_pass=2, **env_kw):
    '''Width-envelope fin removal: midline (Isomap, few knots so fins pull it less) -> smooth body width
       envelope -> drop pixels outside it. Repeated n_pass times, each time measuring the envelope on the
       ORIGINAL mask around the midline of the previous cleaned mask. Returns the cleaned mask.'''
    clean = mask
    for _ in range(n_pass):
        curve = fish_midline_isomap(clean, head, n_knots=6) * (np.array(mask.shape) - 1)
        env_l, env_r, _, _ = body_envelope(mask, curve, **env_kw)
        clean = clip_to_envelope(mask, curve, env_l, env_r)
    return clean

In [ ]:
FIN_FRAMES = [(285, 1), (55, 1), (740, 1), (890, 1), (0, 0), (340, 0), (295, 0), (500, 2)]
def chin_of(fr, f):
    k = rm.set_index(['frame', 'fish']).loc[(fr, f), 'csv_fish']; return keypoint(df.loc[fr + 1], 'chin', k)

fig, axs = plt.subplots(2, 4, figsize=(19, 9))
for ax, (fr, f) in zip(axs.flat, FIN_FRAMES):
    m = np.asarray(M[f, fr]); c, ms = timed(remove_fins, m, chin_of(fr, f))
    img = np.zeros(m.shape + (3,)); img[m & c] = 1; img[m & ~c] = (1, .25, .25)
    pix = np.argwhere(m); lo, hi = pix.min(0) - 8, pix.max(0) + 8
    ax.imshow(img[lo[0]:hi[0], lo[1]:hi[1]], extent=(lo[1] - .5, hi[1] - .5, hi[0] - .5, lo[0] - .5))
    ax.set_title(f'frame {fr} fish {f}: {(m & ~c).sum()} of {m.sum()} px removed (red), {ms:.0f} ms', fontsize=9)
plt.tight_layout(); plt.show()

### 15b. What the real fins look like

Every removed piece of ≥ 25 px that touches the body is one fin. Per piece: area, how far it sticks
out (`protrusion`), mean width (area / protrusion), where it sits (fraction of body length from the
snout), its angle to the body (90° = perpendicular), and whether it points backward (+1) or
forward (−1). This calibrates the synthetic fins in §16.

In [ ]:
from scipy.ndimage import distance_transform_edt
CLEAN, CLEAN_MS, fin_rows, n_fins = {}, {}, [], []
for _, row in rm.iterrows():
    m = np.asarray(M[row.fish, row.frame]); chin = keypoint(df.loc[row.frame + 1], 'chin', row.csv_fish)
    c, CLEAN_MS[(row.frame, row.fish)] = timed(remove_fins, m, chin); CLEAN[(row.frame, row.fish)] = c
    mid = to_px(fish_midline_isomap(c, chin), P)
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(mid, axis=0), axis=1))]; L = s[-1]
    tan = np.gradient(mid, axis=0); tan /= np.linalg.norm(tan, axis=1, keepdims=True)
    dout = distance_transform_edt(~c)
    lab, n = label(m & ~c, structure=np.ones((3, 3))); nf = 0
    for i in range(1, n + 1):
        q = np.argwhere(lab == i); base = q[dout[tuple(q.T)] <= 1.5]
        if len(q) < 25 or len(base) == 0: continue
        nf += 1
        j = cKDTree(mid).query(base.mean(0))[1]; tip = q[np.argmax(dout[tuple(q.T)])]
        v = (tip - base.mean(0)) / np.linalg.norm(tip - base.mean(0)); prot = dout[tuple(q.T)].max()
        fin_rows.append(dict(fish=row.fish, area=len(q), protrusion=prot, width=len(q) / prot, pos_from_head=1 - s[j] / L,
                             angle_to_body=np.degrees(np.arccos(min(1, abs(v @ tan[j])))), points_back=np.sign(-v @ tan[j]),
                             body_len=L))
    n_fins.append(dict(fish=row.fish, n_fins=nf, body_len=L, body_width=2 * distance_transform_edt(c).max()))
fins_df, nfin_df = pd.DataFrame(fin_rows), pd.DataFrame(n_fins)
print('fins per mask:'); display(nfin_df.groupby('fish').n_fins.value_counts().unstack(fill_value=0))
print('body length and max width (px), median:'); display(nfin_df.groupby('fish')[['body_len', 'body_width']].median().round(1))
fins_df['protrusion / L'] = fins_df.protrusion / fins_df.body_len; fins_df['width / L'] = fins_df.width / fins_df.body_len
print('fin geometry, median per fish:')
display(fins_df.groupby('fish')[['area', 'protrusion', 'protrusion / L', 'width / L', 'pos_from_head', 'angle_to_body', 'points_back']].median().round(3))

## 16. Synthetic fish with fins, and every method with vs without fin removal

`make_fish` gets an optional `fins` argument (without it the body is pixel-for-pixel the same as
before). `FINS` draws two pectoral fins matched to §15b: 30–31 % of the length behind the snout,
pointing backward at 55° and 72°. The drawn size (0.12 L and 0.10 L long, 0.045 L wide) is chosen so
that after `remove_fins` the removed area and protrusion match the real median (89 vs 91 px per fin at
L = 180; protrusion 0.092 vs 0.089 L). The two fins differ, so their pulls on the midline don't
cancel. The true midline is unchanged by the fins, so the synthetic test measures each method's fin error directly.

In [ ]:
# pectoral fins, in body lengths / degrees, from the real masks (median over fish 0-3, see measure_fins.py):
# 28-32 % of the length behind the snout, protrude ~0.09 L, ~0.035 L wide, 60-70 deg to the body, pointing back.
# length/width below are the drawn fin; they reproduce the real removed area (89 vs 91 px at L = 180) and protrusion
# (0.092 vs 0.089 L) after remove_fins. The two fins differ so their pulls on the midline don't cancel.
FINS = [dict(pos_from_head=0.30, length=0.12, width=0.045, angle=55, side=+1),
        dict(pos_from_head=0.31, length=0.10, width=0.045, angle=72, side=-1)]


def make_fish(theta_deg, bend, kind='C', center=(456.3, 456.7), P=PS, fins=()):
    s = np.linspace(0, Lf, 2000)
    if kind == 'C':
        kappa = np.deg2rad(bend) / Lf * np.ones_like(s)
    else:
        kappa = np.deg2rad(bend) / Lf * 2 * np.cos(2 * np.pi * s / Lf)
    ang = np.concatenate([[0], np.cumsum(kappa[:-1] * np.diff(s))]) + np.deg2rad(theta_deg)
    ds = np.diff(s, prepend=0)
    mid = np.column_stack([np.cumsum(np.cos(ang) * ds), np.cumsum(np.sin(ang) * ds)])
    mid += np.asarray(center) - mid.mean(0)
    u = s / Lf
    half = 1.5 + 9.0 * np.clip(u / 0.75, 0, 1) ** 1.2
    half = np.where(u > 0.85, half * np.sqrt(np.clip((1 - u) / 0.15, 0, 1)) + 1e-9, half)
    disks = list(zip(mid[::4], half[::4]))
    tan = np.column_stack([np.cos(ang), np.sin(ang)]); nrm = np.column_stack([-tan[:, 1], tan[:, 0]])
    for fin in fins:                                   # a fin = tapering chain of disks from the body edge outward
        i = int(round((1 - fin['pos_from_head']) * (len(s) - 1)))
        a = np.deg2rad(fin['angle'])
        d = np.cos(a) * -tan[i] + np.sin(a) * fin['side'] * nrm[i]       # backward, at `angle` to the body
        base = mid[i] + fin['side'] * nrm[i] * 0.8 * half[i]
        L_fin = fin['length'] * Lf + 0.2 * half[i]
        for f in np.linspace(0, 1, 60):
            disks.append((base + f * L_fin * d, 0.5 * fin['width'] * Lf * (1 - 0.5 * f)))
    pts = np.array([p for p, _ in disks])
    lo, hi = pts.min(0).astype(int) - 15, pts.max(0).astype(int) + 15
    gx, gy = np.mgrid[lo[0]:hi[0], lo[1]:hi[1]]
    sub = np.zeros(gx.shape, bool)
    for (mx, my), h in disks:
        sub |= (gx - mx) ** 2 + (gy - my) ** 2 <= h ** 2
    mask = np.zeros((P, P), bool); mask[lo[0]:hi[0], lo[1]:hi[1]] = sub
    return mid, mask

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(19, 4.8))
for ax, (kind, bend) in zip(axs, [('C', 0), ('C', 150), ('S', 120), ('C', 270)]):
    mid, mask = make_fish(30, bend, kind, fins=FINS); c = remove_fins(mask, mid[-1])
    img = np.zeros(mask.shape + (3,)); img[mask & c] = 1; img[mask & ~c] = (1, .25, .25)
    pix = np.argwhere(mask); lo, hi = pix.min(0) - 8, pix.max(0) + 8
    ax.imshow(img[lo[0]:hi[0], lo[1]:hi[1]], extent=(lo[1] - .5, hi[1] - .5, hi[0] - .5, lo[0] - .5))
    ax.plot(mid[:, 1], mid[:, 0], 'c--', lw=1); ax.set_title(f'{kind}{bend} with fins (red = removed)', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
MASK_METHODS = {k: METHODS[k] for k in ('PCA', 'trace', 'pcurve', 'isomap')}
rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, nofin = make_fish(theta, bend, kind); _, fin = make_fish(theta, bend, kind, fins=FINS)
        for variant, m in (('no fins', nofin), ('fins', fin), ('fins removed', remove_fins(fin, mid[-1]))):
            for name, f in MASK_METHODS.items():
                cur = to_px(f(m, None, mid[-1]), PS)
                rows.append(dict(shape=f'{kind}{bend}', rot=theta, mask=variant, method=name, **score(cur, mid)))
fin_syn = pd.DataFrame(rows)
for col, agg in (('mean_err', 'mean'), ('max_err', 'max')):
    print(f'{col} ({agg} over 3 rotations)')
    display(fin_syn.pivot_table(index='shape', columns=['method', 'mask'], values=col, aggfunc=agg, sort=False)
            .reindex(columns=pd.MultiIndex.from_product([list(MASK_METHODS), ['no fins', 'fins', 'fins removed']])).round(2))

### 16b. Real masks with vs without fin removal

Same 580 masks; `fins removed` uses the masks cleaned in §15b. `inside` is measured against the
**original** mask.

In [ ]:
def real_metrics(cur, m, chin, tail):
    return dict(tail_end_err=np.linalg.norm(cur[0] - tail), head_end_err=np.linalg.norm(cur[-1] - chin),
                inside=inside_frac(m, cur), turning_deg=turning_deg(cur))

def real_summary(d, by):
    g = d.groupby(by, sort=False)
    return pd.DataFrame({'tail end > 20 px': g.tail_end_err.apply(lambda v: (v > 20).sum()), 'tail end median px': g.tail_end_err.median(),
                         '< 95% inside': g.inside.apply(lambda v: (v < .95).sum()), 'turning > 360°': g.turning_deg.apply(lambda v: (v > 360).sum()),
                         'turning > 180°': g.turning_deg.apply(lambda v: (v > 180).sum()), 'median turning °': g.turning_deg.median(),
                         'ms median': g.ms.median()}).round(1)

rows, CURVES = [], {}
for _, row in rm.iterrows():
    key = (row.frame, row.fish); m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]
    chin, tail = keypoint(r, 'chin', row.csv_fish), keypoint(r, 'tail', row.csv_fish)
    for variant, mm in (('fins', m), ('fins removed', CLEAN[key])):
        for name, f in MASK_METHODS.items():
            cur, ms = timed(f, mm, None, chin); cur = to_px(cur, P); CURVES[key + (name, variant)] = cur
            rows.append(dict(frame=row.frame, fish=row.fish, method=name, mask=variant, ms=ms, **real_metrics(cur, m, chin, tail)))
fin_real = pd.DataFrame(rows)
display(real_summary(fin_real, ['method', 'mask']))

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(19, 9.5))
for ax, (fr, f) in zip(axs.flat, FIN_FRAMES):
    show_real(ax, fr, f, curves=[(CURVES[(fr, f, n, 'fins')], STYLE[n][0] + ':', f'{n} (fins)') for n in ('trace', 'pcurve', 'isomap')]
                              + [(CURVES[(fr, f, n, 'fins removed')], STYLE[n], f'{n} (removed)') for n in ('trace', 'pcurve', 'isomap')])
plt.tight_layout(); plt.show()

## 17. Limiting zig-zags

`smooth_midline` refits any midline as a P-spline of arc length (40 knots, ends pinned) with a
penalty on the `order`-th differences of the coefficients. **Order 2** penalises bending itself, so it
also straightens real bends. **Order 3** penalises *changes* in bending: a steady C-bend costs little and wiggles
cost a lot. It is applied after the method, on the fin-removed masks.

Synthetic fish with fins removed have a known true turning (the bend), so `excess turning` = midline
turning − true turning measures leftover wiggle (negative = real bend straightened). The principal
curve's C270/C320 failures (its straight-line start) are left out of its synthetic numbers.

In [ ]:
from scipy.interpolate import BSpline


def smooth_midline(curve, lam=1e-2, order=3, n_knots=40, n_out=None):
    '''Refit a midline (n,2) as a penalised spline (P-spline) of arc length with ends pinned.
       The penalty is on `order`-th differences of the spline coefficients: order 2 penalises bending
       itself (straightens real bends too), order 3 penalises *changes* in bending, so a steady C-bend
       costs little while wiggles cost a lot. lam is relative to the number of points.'''
    n = len(curve) if n_out is None else n_out
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(curve, axis=0), axis=1))]; u = s / s[-1]
    t = np.r_[[0.0] * 4, np.linspace(0, 1, n_knots + 2)[1:-1], [1.0] * 4]
    B = BSpline.design_matrix(u, t, 3).toarray()
    W = np.ones(len(u)); W[[0, -1]] = 1e4                      # pin the ends
    D = np.diff(np.eye(B.shape[1]), order, axis=0)
    c = np.linalg.solve(B.T @ (W[:, None] * B) + lam * len(u) * D.T @ D, B.T @ (W[:, None] * curve))
    dense = BSpline(t, c, 3)(np.linspace(0, 1, 20 * n))
    sd = np.r_[0, np.cumsum(np.linalg.norm(np.diff(dense, axis=0), axis=1))]
    ue = np.linspace(0, sd[-1], n)
    return np.column_stack([np.interp(ue, sd, dense[:, 0]), np.interp(ue, sd, dense[:, 1])])

In [ ]:
SMOOTH_METHODS = ('trace', 'pcurve', 'isomap')
syn_cache = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, fin = make_fish(theta, bend, kind, fins=FINS); c = remove_fins(fin, mid[-1])
        syn_cache.append(dict(shape=f'{kind}{bend}', mid=mid, true_turn=turning_deg(mid[::10]),
                              curves={n: to_px(METHODS[n](c, None, mid[-1]), PS) for n in SMOOTH_METHODS}))
rows = []
for order, lam in [(None, 0), (2, 0.1), (2, 0.3), (2, 1), (3, 1), (3, 3), (3, 10), (3, 30), (3, 100)]:
    sm = (lambda c: c) if order is None else (lambda c: smooth_midline(c, lam, order))
    for n in SMOOTH_METHODS:
        e, ex = [], []
        for s in syn_cache:
            if n == 'pcurve' and s['shape'] in ('C270', 'C320'): continue
            c = sm(s['curves'][n]); sc = score(c, s['mid']); e.append((sc['mean_err'], sc['max_err'])); ex.append(turning_deg(c) - s['true_turn'])
        tr = np.array([turning_deg(sm(CURVES[(fr, f, n, 'fins removed')])) for fr, f in zip(rm.frame, rm.fish)])
        e = np.array(e)
        rows.append({'order': 'none' if order is None else order, 'lam': lam, 'method': n, 'syn mean err': e[:, 0].mean(),
                     'syn worst max err': e[:, 1].max(), 'syn excess turning median °': np.median(ex), 'syn excess turning max °': np.max(ex),
                     'real median turning °': np.median(tr), 'real turning > 180°': (tr > 180).sum(), 'real turning > 360°': (tr > 360).sum()})
display(pd.DataFrame(rows).set_index(['order', 'lam', 'method']).round(2))
SMOOTH = dict(lam=10, order=3)

**Chosen: order 3, `lam = 10`** (`SMOOTH`). The synthetic errors stay as they were, the leftover turning on synthetic
fish is ~0–4° (median), and on real masks the median turning falls from 76–103° to ~41–45°. Order 2 starts
straightening real bends (negative excess turning) before the real-mask turning gets as low.

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(19, 9.5))
for ax, (fr, f) in zip(axs.flat, FIN_FRAMES):
    show_real(ax, fr, f, curves=[(CURVES[(fr, f, n, 'fins removed')], STYLE[n][0] + ':', f'{n}') for n in SMOOTH_METHODS]
                              + [(smooth_midline(CURVES[(fr, f, n, 'fins removed')], **SMOOTH), STYLE[n], f'{n} smoothed') for n in SMOOTH_METHODS])
plt.tight_layout(); plt.show()

## 18. Mask-based Denis: keypoints from the mask, 4 vs 8 points

`fish_midline_mask_denis` keeps Denis's interpolation, oversampling and arc-length picking, but takes
the points from the (fin-removed) mask. Every pixel gets its distance *through the body* from the chin
(one shortest-path search); the farthest pixel is the tail tip. A point at a given position along
the body is the centroid of the pixels at that distance (± 1 px), a slice across the body.
With 4 points they sit at Denis's tail / middle / head / mouth positions (the median positions of
the tracked keypoints); with 8 or 12 they are evenly spaced.

In [ ]:
from scipy.sparse.csgraph import dijkstra

DENIS_FRAC = (0.0, 0.548, 0.894, 1.0)        # tail, middle, head, mouth along tail -> mouth (median of tracked keypoints)


def mask_keypoints(mask, head, fractions=DENIS_FRAC, band=1.0):
    '''Ordered points along the body taken from the mask instead of tracking. Every pixel gets its distance
       through the body from the head (shortest path inside the mask); the farthest pixel is the tail
       tip, L its distance. The point at fraction f (0 = tail, 1 = head, as in Denis's order) is the
       centroid of the pixels at distance (1 - f) * L +- band. f = 1 is the head pixel itself; f = 0 the
       centroid of the pixels within `band` of the farthest distance. Returns (len(fractions), 2) px.'''
    G, pix = _pixel_graph(_largest_component(mask))
    start = np.argmin(np.linalg.norm(pix - head, axis=1))
    g = dijkstra(G, indices=start)
    L = g.max()
    out = []
    for f in fractions:
        target = (1 - f) * L
        if f >= 1: out.append(pix[start]); continue
        sel = np.abs(g - min(target, L - band)) <= band
        out.append(pix[sel].mean(0))
    return np.array(out)


def fish_midline_mask_denis(mask, head, n_points=4, n_ctrl=200):
    '''Denis's method with the keypoints taken from the mask: n_points = 4 uses his tail/middle/head/mouth
       positions; any other n_points uses evenly spaced positions. Same interpolation, oversampling and
       arc-length picking as fish_midline_denis. Returns (n_ctrl,2) in [0,1], tail -> head.'''
    fr = DENIS_FRAC if n_points == 4 else np.linspace(0, 1, n_points)
    return fish_midline_denis(mask_keypoints(mask, head, fr), mask.shape, n_ctrl=n_ctrl)

In [ ]:
rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, fin = make_fish(theta, bend, kind, fins=FINS); c = remove_fins(fin, mid[-1])
        for name, f in (('denis (true keypoints)', lambda: fish_midline_denis(synth_keypoints(mid), fin.shape)),
                        ('mask-denis 4', lambda: fish_midline_mask_denis(c, mid[-1], 4)),
                        ('mask-denis 8', lambda: fish_midline_mask_denis(c, mid[-1], 8)),
                        ('mask-denis 12', lambda: fish_midline_mask_denis(c, mid[-1], 12))):
            cur, ms = timed(f); rows.append(dict(shape=f'{kind}{bend}', method=name, ms=ms, **score(to_px(cur, PS), mid)))
md_syn = pd.DataFrame(rows)
for col, agg in (('mean_err', 'mean'), ('max_err', 'max')):
    print(col); display(md_syn.pivot_table(index='shape', columns='method', values=col, aggfunc=agg, sort=False).round(2))
display(md_syn.groupby('method')[['tail_end_err', 'head_end_err', 'ms']].median().round(2))

## 19. Final comparison

Each method in its best form from §15–18, against the current pipeline function:

| name | what |
|---|---|
| `PCA` | current pipeline function, raw mask |
| `denis` | Denis's method, tracked keypoints (synthetic: keypoints on the true midline) |
| `trace+` / `pcurve+` / `isomap+` | fins removed, then `smooth_midline(order 3, lam 10)` |
| `mask-denis 8` | fins removed, 8 points from the mask |

The fin-removal time (measured per mask) is included in the times of the `+` methods and `mask-denis 8`.

In [ ]:
def final_methods(mask, head, kp, clean=None):
    '''clean = (cleaned mask, ms it took) if already computed, else fins are removed here'''
    c, t_fin = timed(remove_fins, mask, head) if clean is None else clean
    out = {}
    for name, f in (('PCA', lambda: fish_midline_from_mask(mask)),
                    ('denis', lambda: fish_midline_denis(kp, mask.shape)),
                    ('trace+', lambda: smooth_midline(fish_midline_trace(c, head), **SMOOTH)),
                    ('pcurve+', lambda: smooth_midline(fish_midline_pcurve(c, head), **SMOOTH)),
                    ('isomap+', lambda: smooth_midline(fish_midline_isomap(c, head), **SMOOTH)),
                    ('mask-denis 8', lambda: fish_midline_mask_denis(c, head, 8))):
        cur, ms = timed(f); out[name] = (cur, ms + (0 if name in ('PCA', 'denis') else t_fin))
    return out
FINAL = ['PCA', 'denis', 'trace+', 'pcurve+', 'isomap+', 'mask-denis 8']
FSTYLE = dict(zip(FINAL, ['y-', 'm-', 'r-', 'b-', 'g-', 'c-']))

rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, fin = make_fish(theta, bend, kind, fins=FINS)
        for name, (cur, ms) in final_methods(fin, mid[-1], synth_keypoints(mid)).items():
            cur = to_px(cur, PS); rows.append(dict(shape=f'{kind}{bend}', method=name, ms=ms, turning_excess=turning_deg(cur) - turning_deg(mid[::10]), **score(cur, mid)))
final_syn = pd.DataFrame(rows)
print('synthetic fish WITH fins: mean error (px)'); display(final_syn.pivot_table(index='shape', columns='method', values='mean_err', sort=False)[FINAL].round(2))
print('max error (px), worst of 3 rotations'); display(final_syn.pivot_table(index='shape', columns='method', values='max_err', aggfunc='max', sort=False)[FINAL].round(1))
display(final_syn.groupby('method', sort=False)[['tail_end_err', 'head_end_err', 'turning_excess', 'ms']].median().loc[FINAL].round(1))

In [ ]:
rows, FCURVES = [], {}
for _, row in rm.iterrows():
    key = (row.frame, row.fish); m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
    chin, tail = keypoint(r, 'chin', k), keypoint(r, 'tail', k)
    kp = np.array([keypoint(r, p, k) for p in ('tail', 'middle', 'head', 'mouth')])
    for name, (cur, ms) in final_methods(m, chin, kp, clean=(CLEAN[key], CLEAN_MS[key])).items():
        if name == 'denis' and np.isnan(kp).any(): continue
        cur = to_px(cur, P); FCURVES[key + (name,)] = cur
        rows.append(dict(frame=row.frame, fish=row.fish, method=name, ms=ms, **real_metrics(cur, m, chin, tail)))
final_real = pd.DataFrame(rows)
display(real_summary(final_real, 'method').loc[FINAL])

med = pd.DataFrame(np.nan, index=FINAL, columns=FINAL)
for i, a in enumerate(FINAL):
    for j, b in enumerate(FINAL):
        if i == j: continue
        d = np.array([dist_to_poly(FCURVES[key + (a,)], FCURVES[key + (b,)]).mean() for key in zip(rm.frame, rm.fish)
                      if key + (a,) in FCURVES and key + (b,) in FCURVES])
        med.loc[a, b] = np.median(d) if i > j else (d > 5).sum()
print('pairwise: median mean distance in px (lower triangle), masks differing by > 5 px on average (upper triangle)')
display(med.round(2))

In [ ]:
def final_row(keys, title):
    keys = list(keys)[:4]
    fig, axs = plt.subplots(1, 4, figsize=(19, 4.8))
    for ax, (fr, f) in zip(axs, keys):
        show_real(ax, fr, f, curves=[(FCURVES[(fr, f, n)], FSTYLE[n], n) for n in FINAL if (fr, f, n) in FCURVES])
    for ax in axs[len(keys):]: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

final_row(FIN_FRAMES[:4], 'fin frames (fish 1)')
final_row(FIN_FRAMES[4:], 'fin frames (fish 0) and a curled fish 2')
iso_t = final_real[final_real.method == 'isomap+'].set_index(['frame', 'fish']).turning_deg.sort_values(ascending=False)
final_row(iso_t.index, 'most bent fish (by isomap+)')
spread = {key: max(dist_to_poly(FCURVES[key + (a,)], FCURVES[key + (b,)]).mean() for a in FINAL[2:] for b in FINAL[2:] if a != b)
          for key in zip(rm.frame, rm.fish)}
final_row(sorted(spread, key=spread.get, reverse=True), 'largest disagreement among trace+ / pcurve+ / isomap+ / mask-denis 8')

## 20. Five approaches head to head

| name | mask used | method |
|---|---|---|
| `isomap+ (fins removed)` | fin-removed | Isomap midline, then `smooth_midline` (order 3, lam 10) |
| `mask-denis 8 (fins removed)` | fin-removed | 8 slice centres from the mask → Denis's interpolation |
| `isomap+ (raw)` | raw | as above, no fin removal |
| `mask-denis 8 (raw)` | raw | as above, no fin removal |
| `denis (4 SLEAP keypoints)` | none | Denis's method on the tracked tail / middle / head / mouth |

Why these: with fins removed, `isomap+` and `mask-denis 8` had the fewest real-data failures in §19
(`trace+` lost its tail end on 13 masks, `pcurve+` fails on strong curls). On raw masks the same two
are best: `trace+` is the most accurate on synthetic fish but on real raw masks the fins derail
its tail end in 104/580 masks, `pcurve+` zig-zags (37 masks turn > 180°), PCA leaves the body (95 masks < 95 % inside).
Times include fin removal where it is used.

In [ ]:
FIVE = ['isomap+ (fins removed)', 'mask-denis 8 (fins removed)', 'isomap+ (raw)', 'mask-denis 8 (raw)', 'denis (4 SLEAP keypoints)']
F5STYLE = dict(zip(FIVE, ['g-', 'c-', 'g--', 'c--', 'm-']))

def five_methods(mask, head, kp, clean=None):
    '''{name: (curve in [0,1], ms)}; clean = (cleaned mask, ms) if already computed'''
    c, t_fin = timed(remove_fins, mask, head) if clean is None else clean
    out = {}
    for name, m, f, extra in (('isomap+ (fins removed)', c, lambda m: smooth_midline(fish_midline_isomap(m, head) * (P - 1), **SMOOTH) / (P - 1), t_fin),
                              ('mask-denis 8 (fins removed)', c, lambda m: fish_midline_mask_denis(m, head, 8), t_fin),
                              ('isomap+ (raw)', mask, lambda m: smooth_midline(fish_midline_isomap(m, head) * (P - 1), **SMOOTH) / (P - 1), 0),
                              ('mask-denis 8 (raw)', mask, lambda m: fish_midline_mask_denis(m, head, 8), 0),
                              ('denis (4 SLEAP keypoints)', mask, lambda m: fish_midline_denis(kp, m.shape), 0)):
        cur, ms = timed(f, m); out[name] = (cur, ms + extra)
    return out

# synthetic fish with fins
rows = []
for kind, bend in SHAPES_ALL:
    for theta in (0, 30, 73):
        mid, fin = make_fish(theta, bend, kind, fins=FINS)
        for name, (cur, ms) in five_methods(fin, mid[-1], synth_keypoints(mid)).items():
            cur = to_px(cur, PS)
            rows.append(dict(shape=f'{kind}{bend}', method=name, ms=ms, inside=inside_frac(fin, cur),
                             turning_excess=turning_deg(cur) - turning_deg(mid[::10]), **score(cur, mid)))
five_syn = pd.DataFrame(rows); g = five_syn.groupby('method')
print('SYNTHETIC fish with fins (33 fish: 11 shapes x 3 rotations), errors in px vs the true midline')
display(pd.DataFrame({'mean error': g.mean_err.mean(), 'max error: median': g.max_err.median(), 'max error: worst': g.max_err.max(),
                      'tail end error': g.tail_end_err.median(), 'head end error': g.head_end_err.median(),
                      '% of midline inside mask': 100 * g.inside.mean(), 'extra turning vs truth °': g.turning_excess.median()}).loc[FIVE].round(2).T)
print('mean error per shape'); display(five_syn.pivot_table(index='shape', columns='method', values='mean_err', sort=False)[FIVE].round(2))

### 20b. Real masks

`% inside` / `px outside` are measured against the **original** mask (fins included). `interior % inside`
leaves out the 10 points at each end, where a midline legitimately touches the mask edge.
`tail end → tracked tail` and `head end → tracked chin` compare to SLEAP (Denis's tail end *is* the
tracked tail and his head end the tracked mouth). `local shift vs isomap+ (fins removed)`: the largest
distance between that midline and the fins-removed Isomap midline, i.e. how far it strays somewhere along the body.

In [ ]:
rows, CUR5 = [], {}
for _, row in rm.iterrows():
    key = (row.frame, row.fish); m = np.asarray(M[row.fish, row.frame]); r = df.loc[row.frame + 1]; k = row.csv_fish
    chin, tail = keypoint(r, 'chin', k), keypoint(r, 'tail', k)
    kp = np.array([keypoint(r, p, k) for p in ('tail', 'middle', 'head', 'mouth')])
    dout = distance_transform_edt(~m)
    for name, (cur, ms) in five_methods(m, chin, kp, clean=(CLEAN[key], CLEAN_MS[key])).items():
        if name.startswith('denis') and np.isnan(kp).any(): continue
        cur = to_px(cur, P); CUR5[key + (name,)] = cur
        ins = map_coordinates(m.astype(float), cur.T, order=1) >= 0.5
        rows.append(dict(frame=row.frame, fish=row.fish, method=name, ms=ms, inside=ins.mean(), inside_interior=ins[10:-10].mean(),
                         px_out=map_coordinates(dout, cur.T, order=1).max(), length=np.linalg.norm(np.diff(cur, axis=0), axis=1).sum(),
                         **real_metrics(cur, m, chin, tail)))
five_real = pd.DataFrame(rows)
ref = 'isomap+ (fins removed)'
five_real['local_shift'] = [dist_to_poly(CUR5[(fr, f, n)], CUR5[(fr, f, ref)]).max() for fr, f, n in zip(five_real.frame, five_real.fish, five_real.method)]
five_real['mean_shift'] = [dist_to_poly(CUR5[(fr, f, n)], CUR5[(fr, f, ref)]).mean() for fr, f, n in zip(five_real.frame, five_real.fish, five_real.method)]
g = five_real.groupby('method')
cnt = lambda col, thr, op=np.greater: g[col].apply(lambda v: int(op(v, thr).sum()))
print(f'REAL masks: {five_real[["frame", "fish"]].drop_duplicates().shape[0]}')
display(pd.DataFrame({
    '% inside: mean': 100 * g.inside.mean(), '% inside: worst mask': 100 * g.inside.min(), 'interior % inside: mean': 100 * g.inside_interior.mean(),
    'masks 100% inside': cnt('inside', 1, np.greater_equal), 'masks < 99% inside': cnt('inside', .99, np.less),
    'masks < 95% inside': cnt('inside', .95, np.less), 'masks < 80% inside': cnt('inside', .80, np.less),
    'px outside: median': g.px_out.median(), 'px outside: 95th pct': g.px_out.quantile(.95), 'px outside: worst': g.px_out.max(),
    'tail end → tracked tail: median px': g.tail_end_err.median(), 'tail end > 20 px': cnt('tail_end_err', 20),
    'head end → tracked chin: median px': g.head_end_err.median(), 'head end > 20 px': cnt('head_end_err', 20),
    'length: median px': g.length.median(), 'turning: median °': g.turning_deg.median(),
    'turning > 180°': cnt('turning_deg', 180), 'turning > 360°': cnt('turning_deg', 360),
    f'mean shift vs {ref}: median px': g.mean_shift.median(), f'local shift vs {ref}: median px': g.local_shift.median(),
    f'local shift vs {ref}: 95th pct px': g.local_shift.quantile(.95), f'local shift > 5 px (masks)': cnt('local_shift', 5),
    'ms per fish: median': g.ms.median(), 'ms per fish: max': g.ms.max()}).loc[FIVE].round(2).T)

In [ ]:
print('pairwise: median over masks of the mean distance between two midlines (px, lower triangle) and the number of masks where they differ by > 5 px somewhere (upper triangle)')
pw = pd.DataFrame('', index=FIVE, columns=FIVE)
keys = list(zip(rm.frame, rm.fish))
for i, a in enumerate(FIVE):
    for j, b in enumerate(FIVE):
        if i == j: continue
        pairs = [(CUR5[key + (a,)], CUR5[key + (b,)]) for key in keys if key + (a,) in CUR5 and key + (b,) in CUR5]
        pw.loc[a, b] = (f'{np.median([dist_to_poly(x, y).mean() for x, y in pairs]):.2f}' if i > j
                        else str(sum(dist_to_poly(x, y).max() > 5 for x, y in pairs)))
display(pw)
print('by fish: median % inside and median local shift vs isomap+ (fins removed)')
display(five_real.pivot_table(index='fish', columns='method', values='inside', aggfunc=lambda v: 100 * v.mean())[FIVE].round(2))
display(five_real.pivot_table(index='fish', columns='method', values='local_shift', aggfunc='median')[FIVE].round(2))

### 20c. Twenty real fish

Four rows of five masks, each picked by a different criterion (no mask repeats):

1. largest disagreement among all five approaches (in practice: where Denis's keypoint curve leaves the body)
2. largest disagreement among the four mask-based approaches
3. most bent fish (total turning of `isomap+ (fins removed)`)
4. largest fin effect (raw vs fins-removed `mask-denis 8`)

White = body kept after fin removal, red = pixels removed as fin. Solid = fins removed, dashed = raw
mask; green = Isomap, cyan = mask-Denis 8, magenta = Denis with its 4 SLEAP keypoints (magenta dots);
triangle = head end, white dot = tracked chin.

In [ ]:
def show5(ax, fr, f, pad=10):
    m = np.asarray(M[f, fr]); c = CLEAN[(fr, f)]; r = df.loc[fr + 1]
    k = rm.set_index(['frame', 'fish']).loc[(fr, f), 'csv_fish']
    img = np.zeros(m.shape + (3,)); img[c] = 1; img[m & ~c] = (1, .3, .3)
    pix = np.argwhere(m); lo, hi = pix.min(0) - pad, pix.max(0) + pad
    ax.imshow(img[max(lo[0], 0):hi[0], max(lo[1], 0):hi[1]], extent=(max(lo[1], 0) - .5, hi[1] - .5, hi[0] - .5, max(lo[0], 0) - .5))
    for n in FIVE:
        if (fr, f, n) not in CUR5: continue
        cur = CUR5[(fr, f, n)]; st = F5STYLE[n]
        ax.plot(cur[:, 1], cur[:, 0], st, lw=1.4, label=n); ax.plot(cur[-1, 1], cur[-1, 0], st[0] + '^', ms=7)
    kp = np.array([keypoint(r, p, k) for p in ('tail', 'middle', 'head', 'mouth')])
    ax.plot(kp[:, 1], kp[:, 0], 'mo', mec='k', ms=5); ch = keypoint(r, 'chin', k); ax.plot(ch[1], ch[0], 'wo', mec='k', ms=6)
    ax.set_xticks([]); ax.set_yticks([])

def spread_of(names):
    return {key: max(dist_to_poly(CUR5[key + (a,)], CUR5[key + (b,)]).mean() for a in names for b in names
                     if a != b and key + (a,) in CUR5 and key + (b,) in CUR5) for key in keys}

crit = [('largest disagreement, all five', spread_of(FIVE)),
        ('largest disagreement, mask-based four', spread_of(FIVE[:4])),
        ('most bent (turning of isomap+ fins removed)', five_real[five_real.method == FIVE[0]].set_index(['frame', 'fish']).turning_deg.to_dict()),
        ('largest fin effect (mask-denis 8 raw vs fins removed)', {key: dist_to_poly(CUR5[key + (FIVE[3],)], CUR5[key + (FIVE[1],)]).max() for key in keys})]
used, picks = set(), []
for title, score_d in crit:
    chosen = [key for key in sorted(score_d, key=score_d.get, reverse=True) if key not in used][:5]
    used.update(chosen); picks.append((title, chosen, score_d))

fig, axs = plt.subplots(4, 5, figsize=(24, 20))
for row_axs, (title, chosen, score_d) in zip(axs, picks):
    for ax, key in zip(row_axs, chosen):
        show5(ax, *key)
        ax.set_title(f'frame {key[0]}, fish {key[1]}  ({score_d[key]:.1f})', fontsize=9)
    row_axs[0].set_ylabel(title, fontsize=10)
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(rect=(0, 0, 1, 0.975)); plt.show()